# 🎬 YOHA VIDEO · YDC Entertainment Virales — Estudio de generación gratuito (v2)

Convierte la producción creada en la app **Yoha Video** (o una sola instrucción) en clips de video con IA, voces, subtítulos y, opcionalmente, labios sincronizados. Todo con **modelos abiertos** y la **GPU gratuita** de Kaggle o Google Colab. **Costo: US$0** (sin tarjeta, sin APIs de pago).

### Pasos (sin experiencia técnica)
1. **Kaggle** (recomendado, ~30 h de GPU por semana con teléfono verificado): kaggle.com → *Create → New Notebook* → *File → Import Notebook* → este archivo. Panel derecho: **Accelerator → GPU T4 x2**, **Internet → On**.
   **Colab**: colab.research.google.com → *Subir* → *Entorno de ejecución → Cambiar tipo → T4 GPU* (cuota variable).
2. Ejecuta las celdas en orden con ▶. En la **celda 3** pega tu instrucción o sube `yoha-produccion-….json` (pestaña *Crear video → producción → Paquete*).
3. Descarga **`yoha_clips.zip`** al final y ábrelo en Yoha Video → *Editor*. Agrega intro/cierre YDC y música, exporta y envía a revisión.

### Motores (verificados como de uso comercial permitido; revisa sus licencias antes de publicar)
| Etapa | Motor | Licencia | Estado en GPU gratuita |
|---|---|---|---|
| Guion y diálogos | **Qwen2.5-7B-Instruct** (4 bits) | Apache-2.0 | ✔ cabe en T4 |
| Video | **Wan 2.1 T2V 1.3B** (480p, ~5 s) | Apache-2.0 | ✔ T4 |
| Video (mejor calidad) | **Wan 2.2 TI2V 5B** (720p) | Apache-2.0 | ⚠ experimental y lento en T4 |
| Voces ES / EN | **Kokoro 82M** | Apache-2.0 | ✔ CPU/GPU |
| Subtítulos desde el audio | **faster-whisper** (Whisper small) | MIT | ✔ |
| Labios sincronizados | **LatentSync 1.5** | OpenRAIL++ (uso comercial con restricciones de uso) | ⚠ experimental, apagado por defecto |
| Continuidad de personajes | **CLIP ViT-B/32** (parecido con la foto) | MIT | ✔ |
| Escalado 1080p/4K | **FFmpeg lanczos** | LGPL | ✔ (es **escalado**, no 4K nativo) |

Descartados: LivePortrait (sus modelos de detección facial de InsightFace no permiten uso comercial), LTX-Video (licencia no verificada para tu caso), MusicGen (no comercial).

> ⚠️ Realidad: cada clip dura ~5 s y tarda varios minutos en T4. Una novela de 20 min son ~240 clips → varias sesiones de cuota gratuita. El cuaderno **reanuda** donde quedó y **solo regenera las escenas modificadas**. La actuación (emociones, gestos, caminar) se dirige con el texto; el control es aproximado y la consistencia de rostros no es perfecta: el cuaderno la mide y marca escenas a revisar.

## 1 · GPU disponible

In [ ]:
import subprocess, shutil, sys, os, time
T_INICIO = globals().get("T_INICIO") or time.time()
print("Python", sys.version.split()[0])
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout if shutil.which("nvidia-smi") else "❌ Sin GPU: actívala (Kaggle: Accelerator → GPU T4 x2 · Colab: Cambiar tipo → T4 GPU).")
try:
    import torch
    GPU_OK = torch.cuda.is_available()
    VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3 if GPU_OK else 0
    print(f"{'✅' if GPU_OK else '❌'} PyTorch CUDA: {GPU_OK} · VRAM {VRAM_GB:.1f} GB")
except ImportError:
    GPU_OK, VRAM_GB = False, 0

## 2 · Instalar dependencias (3–6 min la primera vez)

In [ ]:
import subprocess, sys
def sh(cmd, check=True):
    print("$", cmd[:120]); r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.returncode and check: print(r.stdout[-1500:], r.stderr[-1500:]); raise SystemExit(f"Falló: {cmd}")
    return r
sh("apt-get -qq update >/dev/null 2>&1; apt-get -qq install -y espeak-ng ffmpeg >/dev/null 2>&1", check=False)
sh(f"{sys.executable} -m pip install -q -U 'diffusers>=0.35' 'transformers>=4.49' accelerate bitsandbytes ftfy imageio imageio-ffmpeg sentencepiece soundfile open_clip_torch faster-whisper requests uv")
# Voces Kokoro: requieren Python < 3.13. Se instalan en un entorno propio con Python 3.12 (uv lo descarga) y torch CPU.
TTS_PY = "/tmp/tts/bin/python"
r = sh("uv venv -q -p 3.12 /tmp/tts && uv pip install -q -p /tmp/tts/bin/python 'kokoro>=0.9.4' soundfile 'pip' "
       "torch --extra-index-url https://download.pytorch.org/whl/cpu --index-strategy unsafe-best-match", check=False)
print("✅ entorno de voces (Python 3.12)" if r.returncode == 0 else "⚠️ Voces no disponibles: " + r.stderr[-400:])
# Voces expresivas: Chatterbox multilingüe (MIT, español, control de emoción). Entorno propio con su torch CUDA.
CBX_PY = "/tmp/cbx/bin/python"
r = sh("uv venv -q -p 3.11 /tmp/cbx && uv pip install -q -p /tmp/cbx/bin/python chatterbox-tts soundfile 'setuptools<81'", check=False)
CBX_OK = r.returncode == 0; print("✅ Chatterbox (voces expresivas)" if CBX_OK else "⚠️ Chatterbox no disponible, se usará Kokoro: " + r.stderr[-300:])
# Música original: ACE-Step v1 3.5B (Apache-2.0). Entorno propio.
ACE_PY = "/tmp/ace/bin/python"
r = sh("uv venv -q -p 3.10 /tmp/ace && uv pip install -q -p /tmp/ace/bin/python git+https://github.com/ace-step/ACE-Step.git", check=False)
ACE_OK = r.returncode == 0; print("✅ ACE-Step (música original)" if ACE_OK else "⚠️ ACE-Step no disponible: " + r.stderr[-300:])
# Labios sincronizados: MuseTalk 1.5 (código y pesos MIT, uso comercial permitido) + control de rostros con MediaPipe (Apache-2.0),
# YuNet (MIT) y SFace (Apache-2.0). Sin S3FD, DWPose ni BiSeNet (licencias dudosas para uso comercial). Entorno propio Python 3.10.
MT_PY, MTM, MT_REPO = "MPLBACKEND=Agg /tmp/mt/bin/python", "/tmp/mtm", "/tmp/MuseTalk"
import base64 as _b64, pathlib as _pl
for _n, _c in {"labios_musetalk.py": "IiIiTGFiaW9zIHNpbmNyb25pemFkb3MgcGFyYSBZb2hhIFZpZGVvIGNvbiBNdXNlVGFsayAxLjUgKGPDs2RpZ28gTUlULCBwZXNvcyBNSVQgwqtwYXJhIGN1YWxxdWllciB1c28sIGluY2x1c28gY29tZXJjaWFswrspLgoKQ29tcG9uZW50ZXMgeSBsaWNlbmNpYXMgKHRvZG9zIGNvbXBhdGlibGVzIGNvbiBtb25ldGl6YWNpw7NuKToKICDCtyBNdXNlVGFsayAxLjUgVU5ldCAoVE1FbHlyYWxhYi9NdXNlVGFsaywgTUlUKSDCtyBWQUUgc2QtdmFlLWZ0LW1zZSAoU3RhYmlsaXR5IEFJLCBNSVQpCiAgwrcgV2hpc3BlciB0aW55IGNvbW8gY29kaWZpY2Fkb3IgZGUgYXVkaW8gKE9wZW5BSSwgQXBhY2hlLTIuMCkgwrcgTWVkaWFQaXBlIEZhY2UgTWVzaCAoR29vZ2xlLCBBcGFjaGUtMi4wKQpOTyBzZSB1c2FuIGxvcyBjb21wb25lbnRlcyBkZSBNdXNlVGFsayBjb24gZHVkYXMgZGUgbGljZW5jaWEgY29tZXJjaWFsOiBuaSBTM0ZELCBuaSBEV1Bvc2UvbW1jdiwgbmkgQmlTZU5ldCAoZW50cmVuYWRvIGNvbgpDZWxlYkFNYXNrLUhRLCBzb2xvIGludmVzdGlnYWNpw7NuKS4gRWwgcmVjb3J0ZSB5IGxhIG3DoXNjYXJhIGRlIGxhIGJvY2Egc2UgY2FsY3VsYW4gY29uIGxvcyBwdW50b3MgZGUgTWVkaWFQaXBlLgoKVXNvOiAgcHl0aG9uIGxhYmlvc19tdXNldGFsay5weSB0cmFiYWpvcy5qc29uCiAgdHJhYmFqb3MuanNvbiA9IHsibXVzZXRhbGsiOiA8cmVwbz4sICJtb2RlbG9zIjogPGRpcj4sICJmcHMiOiAyNCwgInRyYWJham9zIjogW3sidmlkZW8iLCJhdWRpbyIsInNhbGlkYSIsImNhcmEiOiJtYXlvciJ9XX0KQ2FkYSB0cmFiYWpvIGVzY3JpYmUgPHNhbGlkYT4gKHZpZGVvIHNpbiBhdWRpbywgbWlzbW9zIGN1YWRyb3MpIHkgdW4gaW5mb3JtZSBjb24gbGEgY29ycmVsYWNpw7NuIGJvY2HihpR2b3ogYW50ZXMgeSBkZXNwdcOpcy4KIiIiCmltcG9ydCBqc29uLCBtYXRoLCBvcywgc3VicHJvY2Vzcywgc3lzCm9zLmVudmlyb25bIk1QTEJBQ0tFTkQiXSA9ICJBZ2ciICAgIyBLYWdnbGUgaGVyZWRhIGVsIGJhY2tlbmQgZGUgSnVweXRlcjsgTWVkaWFQaXBlIGltcG9ydGEgbWF0cGxvdGxpYgoKaW1wb3J0IGN2MgppbXBvcnQgbnVtcHkgYXMgbnAKaW1wb3J0IHRvcmNoCgojIFB1bnRvcyBkZSBNZWRpYVBpcGUgRmFjZSBNZXNoCk9WQUxPID0gWzEwLCAzMzgsIDI5NywgMzMyLCAyODQsIDI1MSwgMzg5LCAzNTYsIDQ1NCwgMzIzLCAzNjEsIDI4OCwgMzk3LCAzNjUsIDM3OSwgMzc4LCA0MDAsIDM3NywgMTUyLCAxNDgsIDE3NiwgMTQ5LCAxNTAsIDEzNiwKICAgICAgICAgMTcyLCA1OCwgMTMyLCA5MywgMjM0LCAxMjcsIDE2MiwgMjEsIDU0LCAxMDMsIDY3LCAxMDldCk5BUklaLCBOQVJJWl9BTFRBLCBNRU5UT04sIEZSRU5URSwgTEFCX1NVUCwgTEFCX0lORiA9IDE5NSwgNiwgMTUyLCAxMCwgMTMsIDE0CgoKZGVmIGxlZXJfdmlkZW8ocCk6CiAgICBjYXAgPSBjdjIuVmlkZW9DYXB0dXJlKHApOyBmciA9IFtdCiAgICB3aGlsZSBUcnVlOgogICAgICAgIG9rLCBmID0gY2FwLnJlYWQoKQogICAgICAgIGlmIG5vdCBvazogYnJlYWsKICAgICAgICBmci5hcHBlbmQoZikKICAgIGZwcyA9IGNhcC5nZXQoY3YyLkNBUF9QUk9QX0ZQUyk7IGNhcC5yZWxlYXNlKCkKICAgIHJldHVybiBmciwgZnBzCgoKZGVmIGNhcmFzKGZyYW1lcyk6CiAgICAiIiJQdW50b3MgZGUgbGEgY2FyYSBxdWUgaGFibGEgZW4gY2FkYSBjdWFkcm8gKGxhIG3DoXMgZ3JhbmRlOyBzaWd1ZSBhIGxhIG1pc21hIGNhcmEgZW50cmUgY3VhZHJvcykuIiIiCiAgICBpbXBvcnQgbWVkaWFwaXBlIGFzIG1wCiAgICBmbSA9IG1wLnNvbHV0aW9ucy5mYWNlX21lc2guRmFjZU1lc2goc3RhdGljX2ltYWdlX21vZGU9RmFsc2UsIG1heF9udW1fZmFjZXM9MywgcmVmaW5lX2xhbmRtYXJrcz1UcnVlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1pbl9kZXRlY3Rpb25fY29uZmlkZW5jZT0wLjQsIG1pbl90cmFja2luZ19jb25maWRlbmNlPTAuNCkKICAgIG91dCwgcHJldiA9IFtdLCBOb25lCiAgICBmb3IgZiBpbiBmcmFtZXM6CiAgICAgICAgaCwgdyA9IGYuc2hhcGVbOjJdCiAgICAgICAgciA9IGZtLnByb2Nlc3MoY3YyLmN2dENvbG9yKGYsIGN2Mi5DT0xPUl9CR1IyUkdCKSkKICAgICAgICBpZiBub3Qgci5tdWx0aV9mYWNlX2xhbmRtYXJrczoKICAgICAgICAgICAgb3V0LmFwcGVuZChOb25lKTsgY29udGludWUKICAgICAgICBjYW5kcyA9IFtucC5hcnJheShbKHAueCAqIHcsIHAueSAqIGgpIGZvciBwIGluIGxtLmxhbmRtYXJrXSwgbnAuZmxvYXQzMikgZm9yIGxtIGluIHIubXVsdGlfZmFjZV9sYW5kbWFya3NdCiAgICAgICAgaWYgcHJldiBpcyBOb25lOgogICAgICAgICAgICBwdHMgPSBtYXgoY2FuZHMsIGtleT1sYW1iZGEgYzogbnAucHRwKGNbOiwgMF0pICogbnAucHRwKGNbOiwgMV0pKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHB0cyA9IG1pbihjYW5kcywga2V5PWxhbWJkYSBjOiBucC5saW5hbGcubm9ybShjLm1lYW4oMCkgLSBwcmV2Lm1lYW4oMCkpKQogICAgICAgIHByZXYgPSBwdHM7IG91dC5hcHBlbmQocHRzKQogICAgZm0uY2xvc2UoKQogICAgcmV0dXJuIG91dAoKCmRlZiBhcGVydHVyYShwdHMpOgogICAgIiIiQXBlcnR1cmEgZGUgbGEgYm9jYSBub3JtYWxpemFkYSBwb3IgbGEgYWx0dXJhIGRlIGxhIGNhcmEuIiIiCiAgICBpZiBwdHMgaXMgTm9uZTogcmV0dXJuIG5wLm5hbgogICAgcmV0dXJuIGZsb2F0KG5wLmxpbmFsZy5ub3JtKHB0c1tMQUJfU1VQXSAtIHB0c1tMQUJfSU5GXSkgLyBtYXgoMS4wLCBucC5saW5hbGcubm9ybShwdHNbRlJFTlRFXSAtIHB0c1tNRU5UT05dKSkpCgoKZGVmIGNhamEocHRzLCBILCBXLCBtYXJnZW49MTApOgogICAgIiIiUmVjb3J0ZSBpZ3VhbCBhbCBkZSBNdXNlVGFsazogYW5jaG8gZGVsIMOzdmFsbzsgZGUgKG5hcml6IOKIkiBkaXN0YW5jaWEgbmFyaXrihpJtZW50w7NuKSBhbCBtZW50w7NuICsgbWFyZ2VuLiIiIgogICAgb3YgPSBwdHNbT1ZBTE9dCiAgICB4MSwgeDIgPSBvdls6LCAwXS5taW4oKSwgb3ZbOiwgMF0ubWF4KCkKICAgIHkyID0gbWF4KG92WzosIDFdLm1heCgpLCBwdHNbTUVOVE9OXVsxXSk7IHluID0gcHRzW05BUklaXVsxXQogICAgeTEgPSBtYXgoMCwgeW4gLSAoeTIgLSB5bikpCiAgICByZXR1cm4gbnAuYXJyYXkoW21heCgwLCB4MSksIHkxLCBtaW4oVywgeDIpLCBtaW4oSCwgeTIgKyBtYXJnZW4pXSwgbnAuZmxvYXQzMikKCgpkZWYgc3Vhdml6YXIoY2FqYXMsIGs9NSk6CiAgICB2ID0gW2kgZm9yIGksIGMgaW4gZW51bWVyYXRlKGNhamFzKSBpZiBjIGlzIG5vdCBOb25lXQogICAgb3V0ID0gW05vbmVdICogbGVuKGNhamFzKQogICAgZm9yIGkgaW4gdjoKICAgICAgICB3cyA9IFtjYWphc1tqXSBmb3IgaiBpbiByYW5nZShtYXgoMCwgaSAtIGsgLy8gMiksIG1pbihsZW4oY2FqYXMpLCBpICsgayAvLyAyICsgMSkpIGlmIGNhamFzW2pdIGlzIG5vdCBOb25lXQogICAgICAgIG91dFtpXSA9IG5wLm1lZGlhbihucC5zdGFjayh3cyksIDApCiAgICByZXR1cm4gb3V0CgoKZGVmIG1hc2NhcmEocHRzLCBib3gsIHNoYXBlKToKICAgICIiIk3DoXNjYXJhIHN1YXZlIGRlIGxhIG1pdGFkIGluZmVyaW9yIGRlIGxhIGNhcmEgKG1lamlsbGFzIGJhamFzLCBib2NhIHkgbWFuZMOtYnVsYSkgZGVudHJvIGRlIGxhIGNhamEuIiIiCiAgICB4MSwgeTEsIHgyLCB5MiA9IGJveC5hc3R5cGUoaW50KTsgbSA9IG5wLnplcm9zKHNoYXBlWzoyXSwgbnAudWludDgpCiAgICBwb2x5ID0gcHRzW09WQUxPXS5hc3R5cGUobnAuaW50MzIpCiAgICBjdjIuZmlsbFBvbHkobSwgW3BvbHldLCAyNTUpCiAgICBtWzogaW50KHB0c1tOQVJJWl1bMV0pLCA6XSA9IDAgICAgICAgICAgICAgICAjIHNvbG8gcG9yIGRlYmFqbyBkZSBsYSBuYXJpeiAob2pvcyB5IGZyZW50ZSBpbnRhY3RvcykKICAgIG1bOiwgOiB4MV0gPSAwOyBtWzosIHgyOl0gPSAwOyBtW3kyOiwgOl0gPSAwCiAgICBrID0gbWF4KDMsIGludCgwLjA4ICogKHgyIC0geDEpKSAvLyAyICogMiArIDEpCiAgICBtID0gY3YyLmVyb2RlKG0sIG5wLm9uZXMoKDMsIDMpLCBucC51aW50OCksIGl0ZXJhdGlvbnM9bWF4KDEsIGsgLy8gOCkpCiAgICByZXR1cm4gY3YyLkdhdXNzaWFuQmx1cihtLCAoaywgayksIDApLmFzdHlwZShucC5mbG9hdDMyKSAvIDI1NS4wCgoKZGVmIGVudm9sdmVudGUoYXVkaW9fMTZrLCBuLCBmcHMpOgogICAgaG9wID0gMTYwMDAgLyBmcHMKICAgIHJldHVybiBucC5hcnJheShbbnAuc3FydChucC5tZWFuKGF1ZGlvXzE2a1tpbnQoaSAqIGhvcCk6IGludCgoaSArIDEpICogaG9wKV0gKiogMikgKyAxZS0xMikgZm9yIGkgaW4gcmFuZ2UobildKQoKCmRlZiBjb3JyZWxhY2lvbihhLCBlKToKICAgIGEgPSBucC5hc2FycmF5KGEsIGZsb2F0KTsgb2sgPSB+bnAuaXNuYW4oYSkKICAgIGlmIG9rLnN1bSgpIDwgODogcmV0dXJuIE5vbmUKICAgIGJlc3QgPSAtMS4wCiAgICBmb3IgbGFnIGluIHJhbmdlKC0zLCA0KTogICAjIHRvbGVyYSDCsTMgY3VhZHJvcyAo4omIIDAsMTIgcykKICAgICAgICB4ID0gYVttYXgoMCwgbGFnKTogbGVuKGEpICsgbWluKDAsIGxhZyldOyB5ID0gZVttYXgoMCwgLWxhZyk6IGxlbihlKSAtIG1heCgwLCBsYWcpXQogICAgICAgIG0gPSB+bnAuaXNuYW4oeCkKICAgICAgICBpZiBtLnN1bSgpIDwgOCBvciBucC5zdGQoeFttXSkgPCAxZS02IG9yIG5wLnN0ZCh5W21dKSA8IDFlLTk6IGNvbnRpbnVlCiAgICAgICAgYmVzdCA9IG1heChiZXN0LCBmbG9hdChucC5jb3JyY29lZih4W21dLCB5W21dKVswLCAxXSkpCiAgICByZXR1cm4gcm91bmQoYmVzdCwgMykKCgpkZWYgbWFpbihjZmdfcGF0aCk6CiAgICBjZmcgPSBqc29uLmxvYWQob3BlbihjZmdfcGF0aCwgZW5jb2Rpbmc9InV0Zi04IikpCiAgICBzeXMucGF0aC5pbnNlcnQoMCwgY2ZnWyJtdXNldGFsayJdKQogICAgZnJvbSB0cmFuc2Zvcm1lcnMgaW1wb3J0IFdoaXNwZXJNb2RlbAogICAgZnJvbSBtdXNldGFsay5tb2RlbHMudmFlIGltcG9ydCBWQUUKICAgIGZyb20gbXVzZXRhbGsubW9kZWxzLnVuZXQgaW1wb3J0IFVOZXQsIFBvc2l0aW9uYWxFbmNvZGluZwogICAgZnJvbSBtdXNldGFsay51dGlscy5hdWRpb19wcm9jZXNzb3IgaW1wb3J0IEF1ZGlvUHJvY2Vzc29yCiAgICBpbXBvcnQgbGlicm9zYQoKICAgIE0gPSBjZmdbIm1vZGVsb3MiXTsgZGV2ID0gdG9yY2guZGV2aWNlKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICB2YWUgPSBWQUUobW9kZWxfcGF0aD1mIntNfS9zZC12YWUiKTsgdW5ldCA9IFVOZXQodW5ldF9jb25maWc9ZiJ7TX0vbXVzZXRhbGtWMTUvbXVzZXRhbGsuanNvbiIsIG1vZGVsX3BhdGg9ZiJ7TX0vbXVzZXRhbGtWMTUvdW5ldC5wdGgiLCBkZXZpY2U9ZGV2KQogICAgcGUgPSBQb3NpdGlvbmFsRW5jb2RpbmcoZF9tb2RlbD0zODQpCiAgICBkdCA9IHRvcmNoLmZsb2F0MTYgaWYgZGV2LnR5cGUgPT0gImN1ZGEiIGVsc2UgdG9yY2guZmxvYXQzMgogICAgcGUgPSBwZS50byhkZXYsIGR0KTsgdmFlLnZhZSA9IHZhZS52YWUudG8oZGV2LCBkdCk7IHVuZXQubW9kZWwgPSB1bmV0Lm1vZGVsLnRvKGRldiwgZHQpCiAgICBhcCA9IEF1ZGlvUHJvY2Vzc29yKGZlYXR1cmVfZXh0cmFjdG9yX3BhdGg9ZiJ7TX0vd2hpc3BlciIpCiAgICB3aCA9IFdoaXNwZXJNb2RlbC5mcm9tX3ByZXRyYWluZWQoZiJ7TX0vd2hpc3BlciIpLnRvKGRldiwgZHQpLmV2YWwoKQogICAgdDAgPSB0b3JjaC50ZW5zb3IoWzBdLCBkZXZpY2U9ZGV2KQogICAgaW5mb3JtZSA9IFtdCiAgICBmb3Igam9iIGluIGNmZ1sidHJhYmFqb3MiXToKICAgICAgICBmcmFtZXMsIGZwcyA9IGxlZXJfdmlkZW8oam9iWyJ2aWRlbyJdKQogICAgICAgIG4gPSBsZW4oZnJhbWVzKTsgSCwgVyA9IGZyYW1lc1swXS5zaGFwZVs6Ml0KICAgICAgICBkdXIgPSBuIC8gZnBzCiAgICAgICAgIyBhdWRpbyBleGFjdGFtZW50ZSBkZWwgbGFyZ28gZGVsIHBsYW5vICgxNiBrSHogbW9ubykKICAgICAgICBhMTYsIF8gPSBsaWJyb3NhLmxvYWQoam9iWyJhdWRpbyJdLCBzcj0xNjAwMCwgbW9ubz1UcnVlKQogICAgICAgIGExNiA9IG5wLnBhZChhMTYsICgwLCBtYXgoMCwgaW50KGR1ciAqIDE2MDAwKSAtIGxlbihhMTYpKSkpWzogaW50KHJvdW5kKGR1ciAqIDE2MDAwKSldCiAgICAgICAgdG1wd2F2ID0gam9iWyJzYWxpZGEiXSArICIuMTZrLndhdiIKICAgICAgICBpbXBvcnQgc291bmRmaWxlIGFzIHNmOyBzZi53cml0ZSh0bXB3YXYsIGExNiwgMTYwMDApCiAgICAgICAgZmVhdHMsIEwgPSBhcC5nZXRfYXVkaW9fZmVhdHVyZSh0bXB3YXYpCiAgICAgICAgY2h1bmtzID0gYXAuZ2V0X3doaXNwZXJfY2h1bmsoZmVhdHMsIGRldiwgZHQsIHdoLCBMLCBmcHM9ZnBzLCBhdWRpb19wYWRkaW5nX2xlbmd0aF9sZWZ0PTIsIGF1ZGlvX3BhZGRpbmdfbGVuZ3RoX3JpZ2h0PTIpCiAgICAgICAgcHRzID0gY2FyYXMoZnJhbWVzKQogICAgICAgIGNhamFzID0gc3Vhdml6YXIoW2NhamEocCwgSCwgVykgaWYgcCBpcyBub3QgTm9uZSBlbHNlIE5vbmUgZm9yIHAgaW4gcHRzXSkKICAgICAgICBpZHggPSBbaSBmb3IgaSBpbiByYW5nZShuKSBpZiBjYWphc1tpXSBpcyBub3QgTm9uZV0KICAgICAgICByZXMgPSBbZi5jb3B5KCkgZm9yIGYgaW4gZnJhbWVzXQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBsYXQgPSB7fQogICAgICAgICAgICBmb3IgaSBpbiBpZHg6CiAgICAgICAgICAgICAgICB4MSwgeTEsIHgyLCB5MiA9IGNhamFzW2ldLmFzdHlwZShpbnQpCiAgICAgICAgICAgICAgICBjcm9wID0gY3YyLnJlc2l6ZShmcmFtZXNbaV1beTE6eTIsIHgxOngyXSwgKDI1NiwgMjU2KSwgaW50ZXJwb2xhdGlvbj1jdjIuSU5URVJfTEFOQ1pPUzQpCiAgICAgICAgICAgICAgICBsYXRbaV0gPSB2YWUuZ2V0X2xhdGVudHNfZm9yX3VuZXQoY3JvcCkKICAgICAgICAgICAgQiA9IDE2CiAgICAgICAgICAgIGZvciBzIGluIHJhbmdlKDAsIGxlbihpZHgpLCBCKToKICAgICAgICAgICAgICAgIGlpID0gW2kgZm9yIGkgaW4gaWR4W3M6IHMgKyBCXSBpZiBpIDwgbGVuKGNodW5rcyldCiAgICAgICAgICAgICAgICBpZiBub3QgaWk6IGNvbnRpbnVlCiAgICAgICAgICAgICAgICBhZiA9IHBlKHRvcmNoLnN0YWNrKFtjaHVua3NbaV0gZm9yIGkgaW4gaWldKS50byhkZXYsIGR0KSkKICAgICAgICAgICAgICAgIGxiID0gdG9yY2guY2F0KFtsYXRbaV0gZm9yIGkgaW4gaWldLCAwKS50byhkZXYsIGR0KQogICAgICAgICAgICAgICAgcHJlZCA9IHVuZXQubW9kZWwobGIsIHQwLCBlbmNvZGVyX2hpZGRlbl9zdGF0ZXM9YWYpLnNhbXBsZQogICAgICAgICAgICAgICAgZm9yIGksIGZhY2UgaW4gemlwKGlpLCB2YWUuZGVjb2RlX2xhdGVudHMocHJlZCkpOgogICAgICAgICAgICAgICAgICAgIHgxLCB5MSwgeDIsIHkyID0gY2FqYXNbaV0uYXN0eXBlKGludCkKICAgICAgICAgICAgICAgICAgICBmYWNlID0gY3YyLnJlc2l6ZShmYWNlLmFzdHlwZShucC51aW50OCksICh4MiAtIHgxLCB5MiAtIHkxKSwgaW50ZXJwb2xhdGlvbj1jdjIuSU5URVJfTEFOQ1pPUzQpCiAgICAgICAgICAgICAgICAgICAgY2FudmFzID0gZnJhbWVzW2ldLmFzdHlwZShucC5mbG9hdDMyKS5jb3B5KCk7IGNhbnZhc1t5MTp5MiwgeDE6eDJdID0gZmFjZQogICAgICAgICAgICAgICAgICAgIG0gPSBtYXNjYXJhKHB0c1tpXSwgY2FqYXNbaV0sIGZyYW1lc1tpXS5zaGFwZSlbLi4uLCBOb25lXQogICAgICAgICAgICAgICAgICAgIHJlc1tpXSA9IChjYW52YXMgKiBtICsgZnJhbWVzW2ldLmFzdHlwZShucC5mbG9hdDMyKSAqICgxIC0gbSkpLmNsaXAoMCwgMjU1KS5hc3R5cGUobnAudWludDgpCiAgICAgICAgdG1wID0gam9iWyJzYWxpZGEiXSArICIudG1wLm1wNCIKICAgICAgICBwID0gc3VicHJvY2Vzcy5Qb3BlbihbImZmbXBlZyIsICItdiIsICJlcnJvciIsICIteSIsICItZiIsICJyYXd2aWRlbyIsICItcGl4X2ZtdCIsICJiZ3IyNCIsICItcyIsIGYie1d9eHtIfSIsICItciIsIGYie2Zwc30iLCAiLWkiLCAiLSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICItYzp2IiwgImxpYngyNjQiLCAiLXByZXNldCIsICJzbG93IiwgIi1jcmYiLCAiMTQiLCAiLXBpeF9mbXQiLCAieXV2NDIwcCIsIHRtcF0sIHN0ZGluPXN1YnByb2Nlc3MuUElQRSkKICAgICAgICBmb3IgZiBpbiByZXM6IHAuc3RkaW4ud3JpdGUoZi50b2J5dGVzKCkpCiAgICAgICAgcC5zdGRpbi5jbG9zZSgpOyBwLndhaXQoKQogICAgICAgIGlmIHAucmV0dXJuY29kZTogcmFpc2UgUnVudGltZUVycm9yKGYiRkZtcGVnIGZhbGzDsyBhbCBlc2NyaWJpciB7am9iWydzYWxpZGEnXX0iKQogICAgICAgIG9zLnJlcGxhY2UodG1wLCBqb2JbInNhbGlkYSJdKTsgb3MucmVtb3ZlKHRtcHdhdikKICAgICAgICBlbnYgPSBlbnZvbHZlbnRlKGExNiwgbiwgZnBzKQogICAgICAgIGFwMCA9IG5wLmFycmF5KFthcGVydHVyYSh4KSBmb3IgeCBpbiBwdHNdLCBmbG9hdCk7IGFwMSA9IG5wLmFycmF5KFthcGVydHVyYSh4KSBmb3IgeCBpbiBjYXJhcyhyZXMpXSwgZmxvYXQpCiAgICAgICAgaGF5X3ZveiA9IGVudi5tYXgoKSA+IDFlLTMKICAgICAgICB2b3ogPSBlbnYgPiBlbnYubWF4KCkgKiAwLjE1IGlmIGhheV92b3ogZWxzZSBucC56ZXJvcyhuLCBib29sKQogICAgICAgICMgc2lsZW5jaW8gPSBsZWpvcyBkZSBjdWFscXVpZXIgc29uaWRvICjCsTAsMiBzKSwgcGFyYSBubyBjYXN0aWdhciBsYSBib2NhIHF1ZSBzZSBjaWVycmEgYWwgZmluYWwgZGUgdW5hIHBhbGFicmEKICAgICAgICBjZXJjYSA9IG5wLmNvbnZvbHZlKHZvei5hc3R5cGUoZmxvYXQpLCBucC5vbmVzKGludCgwLjQgKiBmcHMpIHwgMSksICJzYW1lIikgPiAwCiAgICAgICAgc2lsID0gfmNlcmNhCiAgICAgICAgbSA9IGxhbWJkYSBhLCBrOiByb3VuZChmbG9hdChucC5uYW5tZWFuKGFba10pKSwgNCkgaWYgay5hbnkoKSBhbmQgbm90IG5wLmlzbmFuKGFba10pLmFsbCgpIGVsc2UgTm9uZQogICAgICAgIHIgPSB7InBsYW5vIjogb3MucGF0aC5iYXNlbmFtZShqb2JbInNhbGlkYSJdKSwgImN1YWRyb3MiOiBuLCAiZnBzIjogcm91bmQoZnBzLCAyKSwgImNhcmFfZGV0ZWN0YWRhIjogcm91bmQobGVuKGlkeCkgLyBtYXgoMSwgbiksIDMpLAogICAgICAgICAgICAgImVzY3VjaGEiOiBib29sKGpvYi5nZXQoImVzY3VjaGEiKSksICJ2b3pfcyI6IHJvdW5kKGZsb2F0KHZvei5tZWFuKCkgKiBkdXIpLCAyKSwKICAgICAgICAgICAgICJjb3JyZWxhY2lvbl9hbnRlcyI6IGNvcnJlbGFjaW9uKGFwMCwgZW52KSBpZiBoYXlfdm96IGVsc2UgTm9uZSwgImNvcnJlbGFjaW9uX2Rlc3B1ZXMiOiBjb3JyZWxhY2lvbihhcDEsIGVudikgaWYgaGF5X3ZveiBlbHNlIE5vbmUsCiAgICAgICAgICAgICAiYXBlcnR1cmFfdm96IjogbShhcDEsIHZveiksICJhcGVydHVyYV9zaWxlbmNpbyI6IG0oYXAxLCBzaWwpLCAiYXBlcnR1cmFfc2lsZW5jaW9fYW50ZXMiOiBtKGFwMCwgc2lsKX0KICAgICAgICBwcmludChqc29uLmR1bXBzKHIsIGVuc3VyZV9hc2NpaT1GYWxzZSksIGZsdXNoPVRydWUpOyBpbmZvcm1lLmFwcGVuZChyKQogICAganNvbi5kdW1wKGluZm9ybWUsIG9wZW4oY2ZnLmdldCgiaW5mb3JtZSIsICJsYWJpb3NfaW5mb3JtZS5qc29uIiksICJ3IiwgZW5jb2Rpbmc9InV0Zi04IiksIGVuc3VyZV9hc2NpaT1GYWxzZSwgaW5kZW50PTEpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oc3lzLmFyZ3ZbMV0pCg==", "control_rostros.py": "IiIiQ29udHJvbCBkZSBpZGVudGlkYWQgeSBlc3RhYmlsaWRhZCBmYWNpYWwgZGUgWW9oYSBWaWRlbyAoc2luIGVtYmVsbGVjZXI6IHNvbG8gTUlERSwgbnVuY2EgYWx0ZXJhIGxhIGltYWdlbikuCgpNb2RlbG9zIChsaWNlbmNpYXMgY29tcGF0aWJsZXMgY29uIHVzbyBjb21lcmNpYWwgZGUgc3VzIGFyY2hpdm9zKToKICDCtyBNZWRpYVBpcGUgRmFjZSBNZXNoIChHb29nbGUsIEFwYWNoZS0yLjApIOKAlCA0NzggcHVudG9zOiBvam9zLCBib2NhLCDDs3ZhbG8uCiAgwrcgWXVOZXQgKE9wZW5DViBab28sIE1JVCkg4oCUIGRldGVjY2nDs24gZGUgY2FyYXMuICDCtyBTRmFjZSAoT3BlbkNWIFpvbywgQXBhY2hlLTIuMCkg4oCUIGh1ZWxsYSBkZSBpZGVudGlkYWQgKGNvc2VubykuCgpQb3IgY2FkYSBwbGFubyBtaWRlLCBjdWFkcm8gYSBjdWFkcm86IGNhcmEgcHJlc2VudGUgwrcgaWRlbnRpZGFkIGZyZW50ZSBhIGxhIHJlZmVyZW5jaWEgZmlqYSBkZWwgcGVyc29uYWplIMK3Cm9qb3MgYXNpbcOpdHJpY29zIMK3IGJvY2EgYW7Ds21hbGEgKGxhYmlvcyBjcnV6YWRvcywgYW5jaHVyYSBpbXBvc2libGUpIMK3IGNhbWJpbyBicnVzY28gZGUgcm9zdHJvIGVudHJlIGN1YWRyb3Mgwrcgbml0aWRlei4KRGV2dWVsdmUgZWwgdmVyZWRpY3RvIGRlbCBwbGFubyB5IGxvcyB0cmFtb3MgZGVmZWN0dW9zb3MgKHBhcmEgcmVnZW5lcmFyIHNvbG8gbG8gcXVlIGZhbGxhKSB5IHVuYSBob2phIGRlIGNvbnRhY3RvIEpQRy4KClVzbzogIHB5dGhvbiBjb250cm9sX3Jvc3Ryb3MucHkgdGFyZWFzLmpzb24gICAgICAodGFyZWFzID0geyJtb2RlbG9zIjogZGlyLCAicmVmcyI6IHtjaWQ6IHBuZ30sICJwbGFub3MiOiBbeyJ2aWRlbyIsImVzcGVyYWRvcyI6W2NpZF0sImlkIn1dLCAic2FsaWRhIjoganNvbn0pCiAgICAgIHB5dGhvbiBjb250cm9sX3Jvc3Ryb3MucHkgLS1lbGVnaXIgY2FuZC5qc29uIChlbGlnZSBsYSBtZWpvciBpbWFnZW4gZGUgcmVmZXJlbmNpYTogY2FyYSBjbGFyYSwgc2ltw6l0cmljYSwgbsOtdGlkYSwgcGFyZWNpZGEgYSBsYSBpZGVudGlkYWQgZmlqYSkKIiIiCmltcG9ydCBqc29uLCBvcywgc3lzCm9zLmVudmlyb25bIk1QTEJBQ0tFTkQiXSA9ICJBZ2ciICAgIyBLYWdnbGUgaGVyZWRhIGVsIGJhY2tlbmQgZGUgSnVweXRlcjsgTWVkaWFQaXBlIGltcG9ydGEgbWF0cGxvdGxpYgoKaW1wb3J0IGN2MgppbXBvcnQgbnVtcHkgYXMgbnAKClVNQlJBTF9JRCA9IDAuMzUgICAgICAgICMgUFJPVklTSU9OQUwgKGNhbGlicmFyIGNvbiBtw6FzIHBsYW5vcyByZWFsZXMpOiByb3N0cm9zIGdlbmVyYWRvcyBwb3IgSUEgZW4gMy80IGRhbiBzaW1pbGl0dWRlcyBTRmFjZSBtw6FzIGJhamFzIHF1ZSBmb3RvcyByZWFsZXMKVU1CUkFMX0lEX0RVUk8gPSAwLjI1ICAgIyBwb3IgZGViYWpvOiBvdHJhIHBlcnNvbmEKT0pPX0kgPSAoMzMsIDEzMywgMTU5LCAxNDUpICAgICMgZXNxdWluYSBleHQuLCBlc3F1aW5hIGludC4sIHDDoXJwYWRvIHN1cC4sIHDDoXJwYWRvIGluZi4gKG9qbyBkZXJlY2hvIGRlIGxhIHBlcnNvbmEpCk9KT19EID0gKDI2MywgMzYyLCAzODYsIDM3NCkKQk9DQSA9ICg2MSwgMjkxLCAxMywgMTQsIDAsIDE3KSAgICMgY29taXN1cmFzLCBsYWJpbyBpbnQuIHN1cC4vaW5mLiwgbGFiaW8gZXh0LiBzdXAuL2luZi4KCgpjbGFzcyBNZWRpZG9yOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIG1vZGVsb3MpOgogICAgICAgIGltcG9ydCBtZWRpYXBpcGUgYXMgbXAKICAgICAgICBzZWxmLm1wID0gbXAKICAgICAgICBzZWxmLmRldCA9IGN2Mi5GYWNlRGV0ZWN0b3JZTi5jcmVhdGUoZiJ7bW9kZWxvc30vZmFjZV9kZXRlY3Rpb25feXVuZXRfMjAyM21hci5vbm54IiwgIiIsICgzMjAsIDMyMCksIDAuNiwgMC4zLCA1MDAwKQogICAgICAgIHNlbGYucmVjID0gY3YyLkZhY2VSZWNvZ25pemVyU0YuY3JlYXRlKGYie21vZGVsb3N9L2ZhY2VfcmVjb2duaXRpb25fc2ZhY2VfMjAyMWRlYy5vbm54IiwgIiIpCgogICAgZGVmIGh1ZWxsYXMoc2VsZiwgaW1nKToKICAgICAgICBoLCB3ID0gaW1nLnNoYXBlWzoyXTsgc2VsZi5kZXQuc2V0SW5wdXRTaXplKCh3LCBoKSkKICAgICAgICBfLCBmID0gc2VsZi5kZXQuZGV0ZWN0KGltZykKICAgICAgICBvdXQgPSBbXQogICAgICAgIGZvciByb3cgaW4gKGYgaWYgZiBpcyBub3QgTm9uZSBlbHNlIFtdKToKICAgICAgICAgICAgb3V0LmFwcGVuZCgocm93LCBzZWxmLnJlYy5mZWF0dXJlKHNlbGYucmVjLmFsaWduQ3JvcChpbWcsIHJvdykpLmZsYXR0ZW4oKSkpCiAgICAgICAgcmV0dXJuIG91dAoKICAgIGRlZiBzaW0oc2VsZiwgYSwgYik6CiAgICAgICAgcmV0dXJuIGZsb2F0KHNlbGYucmVjLm1hdGNoKGEucmVzaGFwZSgxLCAtMSksIGIucmVzaGFwZSgxLCAtMSksIGN2Mi5GYWNlUmVjb2duaXplclNGX0ZSX0NPU0lORSkpCgoKZGVmIGdlb21ldHJpYShwdHMpOgogICAgIiIiSW5kaWNhZG9yZXMgZGUgZGVmb3JtYWNpw7NuIGEgcGFydGlyIGRlIGxvcyBwdW50b3MgZGUgRmFjZSBNZXNoIChpbmRlcGVuZGllbnRlcyBkZWwgdGFtYcOxbyBkZSBsYSBjYXJhKS4iIiIKICAgIGQgPSBsYW1iZGEgYSwgYjogZmxvYXQobnAubGluYWxnLm5vcm0ocHRzW2FdIC0gcHRzW2JdKSkKICAgIGlvID0gZCgzMywgMjYzKSArIDFlLTYgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBkaXN0YW5jaWEgZW50cmUgb2pvcwogICAgYW5jaG9faSwgYW5jaG9fZCA9IGQoT0pPX0lbMF0sIE9KT19JWzFdKSwgZChPSk9fRFswXSwgT0pPX0RbMV0pCiAgICBhbHRvX2ksIGFsdG9fZCA9IGQoT0pPX0lbMl0sIE9KT19JWzNdKSwgZChPSk9fRFsyXSwgT0pPX0RbM10pCiAgICAjIGdpcm8gZGUgY2FiZXphOiBwb3NpY2nDs24gZGUgbGEgbmFyaXogZW50cmUgbG9zIG9qb3MgKDAgPSBkZSBmcmVudGUpCiAgICBnaXJvID0gZmxvYXQoKHB0c1sxXVswXSAtIChwdHNbMzNdWzBdICsgcHRzWzI2M11bMF0pIC8gMikgLyBpbykKICAgIGFzaW1fYW5jaG8gPSBhYnMoYW5jaG9faSAtIGFuY2hvX2QpIC8gbWF4KGFuY2hvX2ksIGFuY2hvX2QpCiAgICBib2NhX2FuY2hvID0gZChCT0NBWzBdLCBCT0NBWzFdKSAvIGlvCiAgICBjcnV6YWRvcyA9IGZsb2F0KChwdHNbQk9DQVsyXV1bMV0gLSBwdHNbQk9DQVszXV1bMV0pIC8gaW8pICAjID4wIHNpIGVsIGxhYmlvIHN1cGVyaW9yIHF1ZWRhIHBvciBkZWJham8gZGVsIGluZmVyaW9yCiAgICByZXR1cm4geyJnaXJvIjogZ2lybywgImFzaW1fb2pvcyI6IGFzaW1fYW5jaG8sICJvam9faSI6IGFsdG9faSAvIGlvLCAib2pvX2QiOiBhbHRvX2QgLyBpbywgImJvY2FfYW5jaG8iOiBib2NhX2FuY2hvLCAibGFiaW9zX2NydXphZG9zIjogY3J1emFkb3N9CgoKZGVmIGZvcm1hKHB0cyk6CiAgICAiIiJGb3JtYSBub3JtYWxpemFkYSAoc2luIHBvc2ljacOzbiBuaSBlc2NhbGEpIHBhcmEgbWVkaXIgY2FtYmlvcyBicnVzY29zIGVudHJlIGN1YWRyb3MuIiIiCiAgICBwID0gcHRzWzo0NjhdIC0gcHRzWzo0NjhdLm1lYW4oMCkKICAgIHJldHVybiBwIC8gKG5wLmxpbmFsZy5ub3JtKHApICsgMWUtOSkKCgpkZWYgbml0aWRleihpbWcsIGJveCk6CiAgICB4LCB5LCB3LCBoID0gW2ludCh2KSBmb3IgdiBpbiBib3hbOjRdXQogICAgYyA9IGltZ1ttYXgoMCwgeSk6IHkgKyBoLCBtYXgoMCwgeCk6IHggKyB3XQogICAgaWYgYy5zaXplID09IDA6IHJldHVybiAwLjAKICAgIGcgPSBjdjIuY3Z0Q29sb3IoY3YyLnJlc2l6ZShjLCAoMTYwLCAxNjApKSwgY3YyLkNPTE9SX0JHUjJHUkFZKQogICAgcmV0dXJuIGZsb2F0KGN2Mi5MYXBsYWNpYW4oZywgY3YyLkNWXzY0RikudmFyKCkpCgoKZGVmIGxlZXIodmlkZW8sIHBhc289MSk6CiAgICBjYXAgPSBjdjIuVmlkZW9DYXB0dXJlKHZpZGVvKTsgZnIgPSBbXTsgaSA9IDAKICAgIHdoaWxlIFRydWU6CiAgICAgICAgb2ssIGYgPSBjYXAucmVhZCgpCiAgICAgICAgaWYgbm90IG9rOiBicmVhawogICAgICAgIGlmIGkgJSBwYXNvID09IDA6IGZyLmFwcGVuZChmKQogICAgICAgIGkgKz0gMQogICAgZnBzID0gY2FwLmdldChjdjIuQ0FQX1BST1BfRlBTKTsgY2FwLnJlbGVhc2UoKQogICAgcmV0dXJuIGZyLCBmcHMKCgpkZWYgYW5hbGl6YXIobWVkLCB2aWRlbywgcmVmcywgZXNwZXJhZG9zLCBwYXNvPTIsIGVzY2VuYT1Ob25lLCB1bmljYT1GYWxzZSk6CiAgICAiIiJNaWRlIHVuIHBsYW5vLiByZWZzID0ge2NsYXZlOiBodWVsbGEgU0ZhY2V9LiBlc3BlcmFkb3MgPSByZWZlcmVuY2lhcyBPRklDSUFMRVMgZGUgcXVpZW5lcyBkZWJlbiBhcGFyZWNlcjsKICAgIGVzY2VuYSA9IHJlZmVyZW5jaWEgZGVsIHByaW1lciBjdWFkcm8gZGUgbGEgZXNjZW5hIChjb25zaXN0ZW5jaWEgZGVudHJvIGRlIGxhIGVzY2VuYSk7IHVuaWNhID0gcHJpbWVyIHBsYW5vIGRlIHVuYSBzb2xhIHBlcnNvbmEuIiIiCiAgICBmcmFtZXMsIGZwcyA9IGxlZXIodmlkZW8sIHBhc28pCiAgICBodWVsbGFzX3BsYW5vID0gW10KICAgIGZtID0gbWVkLm1wLnNvbHV0aW9ucy5mYWNlX21lc2guRmFjZU1lc2goc3RhdGljX2ltYWdlX21vZGU9RmFsc2UsIG1heF9udW1fZmFjZXM9NCwgcmVmaW5lX2xhbmRtYXJrcz1UcnVlLCBtaW5fZGV0ZWN0aW9uX2NvbmZpZGVuY2U9MC40KQogICAgZmlsYXMsIHByZXZfZm9ybWEsIHByZXZfaHVlbGxhID0gW10sIE5vbmUsIE5vbmUKICAgIGNhbmQgPSB7YzogcmVmc1tjXSBmb3IgYyBpbiAoZXNwZXJhZG9zIG9yIHJlZnMpIGlmIGMgaW4gcmVmc30KICAgIGZvciBrLCBmIGluIGVudW1lcmF0ZShmcmFtZXMpOgogICAgICAgIGgsIHcgPSBmLnNoYXBlWzoyXQogICAgICAgIGZpbGEgPSB7InQiOiByb3VuZChrICogcGFzbyAvIChmcHMgb3IgMjQpLCAyKSwgImNhcmEiOiBGYWxzZSwgInByb2JsZW1hcyI6IFtdfQogICAgICAgIGhzID0gbWVkLmh1ZWxsYXMoZikKICAgICAgICBpZiBoczoKICAgICAgICAgICAgYm94LCBodSA9IG1heChocywga2V5PWxhbWJkYSB4OiB4WzBdWzJdICogeFswXVszXSkgICAgICAgICAgICAjIGxhIGNhcmEgcHJpbmNpcGFsIChtw6FzIGdyYW5kZSkKICAgICAgICAgICAgZmlsYVsiY2FyYSJdID0gVHJ1ZTsgZmlsYVsidGFtIl0gPSByb3VuZChmbG9hdChib3hbM10pIC8gaCwgMyk7IGh1ZWxsYXNfcGxhbm8uYXBwZW5kKGh1KQogICAgICAgICAgICBpZiB1bmljYTogICAjIGVuIHVuIHByaW1lciBwbGFubyBubyBwdWVkZSBoYWJlciBvdHJhIGNhcmEgKG5pIGFqZW5hIGVuIGVsIGJvcmRlIG5pIGR1cGxpY2FkYSkKICAgICAgICAgICAgICAgIG90cmFzID0gW2IgZm9yIGIsIF8gaW4gaHMgaWYgYiBpcyBub3QgYm94IGFuZCBiWzNdID4gMC4wOCAqIGhdCiAgICAgICAgICAgICAgICBpZiBvdHJhczogZmlsYVsicHJvYmxlbWFzIl0uYXBwZW5kKCJjYXJhIGFqZW5hIG8gZHVwbGljYWRhIGVuIGVsIGVuY3VhZHJlIikKICAgICAgICAgICAgaWYgY2FuZDoKICAgICAgICAgICAgICAgIHNpbXMgPSB7YzogbWVkLnNpbShodSwgcikgZm9yIGMsIHIgaW4gY2FuZC5pdGVtcygpfQogICAgICAgICAgICAgICAgcXVpZW4gPSBtYXgoc2ltcywga2V5PXNpbXMuZ2V0KTsgZmlsYVsicXVpZW4iXSwgZmlsYVsiaWRlbnRpZGFkIl0gPSBxdWllbiwgcm91bmQoc2ltc1txdWllbl0sIDMpCiAgICAgICAgICAgICAgICBpZiBzaW1zW3F1aWVuXSA8IFVNQlJBTF9JRF9EVVJPOiBmaWxhWyJwcm9ibGVtYXMiXS5hcHBlbmQoInJvc3RybyBkaXN0aW50byBhbCBwZXJzb25hamUgb2ZpY2lhbCIpCiAgICAgICAgICAgICAgICBlbGlmIHNpbXNbcXVpZW5dIDwgVU1CUkFMX0lEOiBmaWxhWyJwcm9ibGVtYXMiXS5hcHBlbmQoInBhcmVjaWRvIGJham8gY29uIGxhIHJlZmVyZW5jaWEgb2ZpY2lhbCIpCiAgICAgICAgICAgIGlmIGVzY2VuYSBhbmQgZXNjZW5hIGluIHJlZnM6CiAgICAgICAgICAgICAgICBzZSA9IG1lZC5zaW0oaHUsIHJlZnNbZXNjZW5hXSk7IGZpbGFbImlkZW50aWRhZF9lc2NlbmEiXSA9IHJvdW5kKHNlLCAzKQogICAgICAgICAgICAgICAgaWYgc2UgPCBVTUJSQUxfSURfRFVSTzogZmlsYVsicHJvYmxlbWFzIl0uYXBwZW5kKCJyb3N0cm8gZGlzdGludG8gYWwgZGVsIGluaWNpbyBkZSBsYSBlc2NlbmEiKQogICAgICAgICAgICBpZiBwcmV2X2h1ZWxsYSBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIGNvbnQgPSBtZWQuc2ltKGh1LCBwcmV2X2h1ZWxsYSk7IGZpbGFbImNvbnRpbnVpZGFkIl0gPSByb3VuZChjb250LCAzKQogICAgICAgICAgICAgICAgaWYgY29udCA8IDAuNDU6IGZpbGFbInByb2JsZW1hcyJdLmFwcGVuZCgiY2FtYmlvIGJydXNjbyBkZSByb3N0cm8iKQogICAgICAgICAgICBwcmV2X2h1ZWxsYSA9IGh1CiAgICAgICAgICAgIGZpbGFbIm5pdGlkZXoiXSA9IHJvdW5kKG5pdGlkZXooZiwgYm94KSwgMSkKICAgICAgICAgICAgciA9IGZtLnByb2Nlc3MoY3YyLmN2dENvbG9yKGYsIGN2Mi5DT0xPUl9CR1IyUkdCKSkKICAgICAgICAgICAgaWYgdW5pY2EgYW5kIHIubXVsdGlfZmFjZV9sYW5kbWFya3MgYW5kIGxlbihyLm11bHRpX2ZhY2VfbGFuZG1hcmtzKSA+IDEgYW5kICJjYXJhIGFqZW5hIG8gZHVwbGljYWRhIGVuIGVsIGVuY3VhZHJlIiBub3QgaW4gZmlsYVsicHJvYmxlbWFzIl06CiAgICAgICAgICAgICAgICBmaWxhWyJwcm9ibGVtYXMiXS5hcHBlbmQoImNhcmEgYWplbmEgbyBkdXBsaWNhZGEgZW4gZWwgZW5jdWFkcmUiKSAgICMgRmFjZSBNZXNoIHRhbWJpw6luIHZlIGNhcmFzIHBhcmNpYWxlcyBlbiBsb3MgYm9yZGVzCiAgICAgICAgICAgIGlmIHIubXVsdGlfZmFjZV9sYW5kbWFya3M6CiAgICAgICAgICAgICAgICB0b2RhcyA9IFtucC5hcnJheShbKHAueCAqIHcsIHAueSAqIGgpIGZvciBwIGluIGxtLmxhbmRtYXJrXSwgbnAuZmxvYXQzMikgZm9yIGxtIGluIHIubXVsdGlfZmFjZV9sYW5kbWFya3NdCiAgICAgICAgICAgICAgICBwdHMgPSBtYXgodG9kYXMsIGtleT1sYW1iZGEgcTogbnAucHRwKHFbOiwgMF0pICogbnAucHRwKHFbOiwgMV0pKQogICAgICAgICAgICAgICAgZyA9IGdlb21ldHJpYShwdHMpOyBmaWxhLnVwZGF0ZSh7azI6IHJvdW5kKHYsIDMpIGZvciBrMiwgdiBpbiBnLml0ZW1zKCl9KQogICAgICAgICAgICAgICAgaWYgYWJzKGdbImdpcm8iXSkgPCAwLjE1IGFuZCBnWyJhc2ltX29qb3MiXSA+IDAuMzU6IGZpbGFbInByb2JsZW1hcyJdLmFwcGVuZCgib2pvcyBhc2ltw6l0cmljb3MiKQogICAgICAgICAgICAgICAgaWYgZ1sibGFiaW9zX2NydXphZG9zIl0gPiAwLjAyOiBmaWxhWyJwcm9ibGVtYXMiXS5hcHBlbmQoImJvY2EgZGVmb3JtYWRhIChsYWJpb3MgY3J1emFkb3MpIikKICAgICAgICAgICAgICAgIGlmIG5vdCAwLjMwIDw9IGdbImJvY2FfYW5jaG8iXSA8PSAxLjMwOiBmaWxhWyJwcm9ibGVtYXMiXS5hcHBlbmQoImJvY2EgZGUgdGFtYcOxbyBhbsOzbWFsbyIpICAgIyBjYWxpYnJhZG86IGJvY2FzIHJlYWxlcyAwLDQz4oCTMCw3NgogICAgICAgICAgICAgICAgc2YgPSBmb3JtYShwdHMpCiAgICAgICAgICAgICAgICBpZiBwcmV2X2Zvcm1hIGlzIG5vdCBOb25lOgogICAgICAgICAgICAgICAgICAgIHNhbHRvID0gZmxvYXQobnAubGluYWxnLm5vcm0oc2YgLSBwcmV2X2Zvcm1hKSk7IGZpbGFbInNhbHRvX2Zvcm1hIl0gPSByb3VuZChzYWx0bywgNCkKICAgICAgICAgICAgICAgICAgICBpZiBzYWx0byA+IDAuMDM1ICogcGFzbzogZmlsYVsicHJvYmxlbWFzIl0uYXBwZW5kKCJkZWZvcm1hY2nDs24gc8O6Yml0YSBkZWwgcm9zdHJvIikKICAgICAgICAgICAgICAgIHByZXZfZm9ybWEgPSBzZgogICAgICAgICAgICBlbHNlOgogICAgICAgICAgICAgICAgcHJldl9mb3JtYSA9IE5vbmUKICAgICAgICBlbHNlOgogICAgICAgICAgICBwcmV2X2h1ZWxsYSA9IHByZXZfZm9ybWEgPSBOb25lCiAgICAgICAgZmlsYXMuYXBwZW5kKGZpbGEpCiAgICBmbS5jbG9zZSgpCiAgICBjb24gPSBbeCBmb3IgeCBpbiBmaWxhcyBpZiB4WyJjYXJhIl1dCiAgICBtYWxvcyA9IFt4IGZvciB4IGluIGZpbGFzIGlmIHhbInByb2JsZW1hcyJdXQogICAgaWRzID0gW3hbImlkZW50aWRhZCJdIGZvciB4IGluIGNvbiBpZiAiaWRlbnRpZGFkIiBpbiB4XQogICAgIyB0cmFtb3MgZGVmZWN0dW9zb3MgY29udGlndW9zIChlbiBzZWd1bmRvcykKICAgIHRyYW1vcywgaW5pID0gW10sIE5vbmUKICAgIGZvciB4IGluIGZpbGFzOgogICAgICAgIGlmIHhbInByb2JsZW1hcyJdIGFuZCBpbmkgaXMgTm9uZTogaW5pID0geFsidCJdCiAgICAgICAgaWYgbm90IHhbInByb2JsZW1hcyJdIGFuZCBpbmkgaXMgbm90IE5vbmU6IHRyYW1vcy5hcHBlbmQoW2luaSwgeFsidCJdXSk7IGluaSA9IE5vbmUKICAgIGlmIGluaSBpcyBub3QgTm9uZTogdHJhbW9zLmFwcGVuZChbaW5pLCBmaWxhc1stMV1bInQiXV0pCiAgICBmcmFjX21hbG9zID0gbGVuKG1hbG9zKSAvIG1heCgxLCBsZW4oZmlsYXMpKQogICAgcmVzID0geyJ2aWRlbyI6IG9zLnBhdGguYmFzZW5hbWUodmlkZW8pLCAiY3VhZHJvc19tZWRpZG9zIjogbGVuKGZpbGFzKSwgImNhcmFfcHJlc2VudGUiOiByb3VuZChsZW4oY29uKSAvIG1heCgxLCBsZW4oZmlsYXMpKSwgMyksCiAgICAgICAgICAgImlkZW50aWRhZF9tZWRpYSI6IHJvdW5kKGZsb2F0KG5wLm1lYW4oaWRzKSksIDMpIGlmIGlkcyBlbHNlIE5vbmUsICJpZGVudGlkYWRfbWluIjogcm91bmQoZmxvYXQobnAubWluKGlkcykpLCAzKSBpZiBpZHMgZWxzZSBOb25lLAogICAgICAgICAgICJjdWFkcm9zX2Nvbl9kZWZlY3RvIjogcm91bmQoZnJhY19tYWxvcywgMyksICJ0cmFtb3NfZGVmZWN0dW9zb3MiOiB0cmFtb3MsCiAgICAgICAgICAgInByb2JsZW1hcyI6IHNvcnRlZCh7cCBmb3IgeCBpbiBtYWxvcyBmb3IgcCBpbiB4WyJwcm9ibGVtYXMiXX0pLCAiZGV0YWxsZSI6IGZpbGFzfQogICAgIyBWZXJlZGljdG86IGlkZW50aWRhZCBlc3RhYmxlIHkg4omkIDggJSBkZSBjdWFkcm9zIGNvbiBkZWZlY3RvcyAodW4gcGFycGFkZW8gcmFybyBhaXNsYWRvIG5vIGNvbmRlbmEgZWwgcGxhbm8pCiAgICBpZGUgPSBbeFsiaWRlbnRpZGFkX2VzY2VuYSJdIGZvciB4IGluIGNvbiBpZiAiaWRlbnRpZGFkX2VzY2VuYSIgaW4geF0KICAgIHJlc1siaWRlbnRpZGFkX2VzY2VuYV9tZWRpYSJdID0gcm91bmQoZmxvYXQobnAubWVhbihpZGUpKSwgMykgaWYgaWRlIGVsc2UgTm9uZQogICAgcmVzWyJfaHVlbGxhIl0gPSAobnAubWVhbihodWVsbGFzX3BsYW5vLCAwKS50b2xpc3QoKSBpZiBodWVsbGFzX3BsYW5vIGVsc2UgTm9uZSkKICAgIHJlc1sicHVudGFqZSJdID0gcm91bmQoKHJlc1siaWRlbnRpZGFkX21lZGlhIl0gb3IgMC4wKSAtIGZyYWNfbWFsb3MsIDMpCiAgICByZXNbImFwcm9iYWRvIl0gPSAoYm9vbChjb24pIGFuZCAobm90IGNhbmQgb3IgKHJlc1siaWRlbnRpZGFkX21lZGlhIl0gb3IgMCkgPj0gVU1CUkFMX0lEKSBhbmQgZnJhY19tYWxvcyA8PSAwLjA4CiAgICAgICAgICAgICAgICAgICAgICAgYW5kIChyZXNbImlkZW50aWRhZF9lc2NlbmFfbWVkaWEiXSBpcyBOb25lIG9yIHJlc1siaWRlbnRpZGFkX2VzY2VuYV9tZWRpYSJdID49IFVNQlJBTF9JRCkpCiAgICByZXR1cm4gcmVzCgoKZGVmIGhvamFfY29udGFjdG8odmlkZW8sIHJlcywgc2FsaWRhLCBuPTgpOgogICAgIiIiSG9qYSBkZSBjb250YWN0byBwYXJhIGxhIHJldmlzacOzbiB2aXN1YWw6IDggY3VhZHJvcyByZXBhcnRpZG9zLCBtYXJjYWRvcyBlbiByb2pvIHNpIHRpZW5lbiBkZWZlY3Rvcy4iIiIKICAgIGZyYW1lcywgZnBzID0gbGVlcih2aWRlbywgMSkKICAgIGlmIG5vdCBmcmFtZXM6IHJldHVybgogICAgaWR4ID0gbnAubGluc3BhY2UoMCwgbGVuKGZyYW1lcykgLSAxLCBuKS5hc3R5cGUoaW50KQogICAgZGV0ID0ge3JvdW5kKHhbInQiXSwgMSk6IHggZm9yIHggaW4gcmVzLmdldCgiZGV0YWxsZSIsIFtdKX0KICAgIHRpbGVzID0gW10KICAgIGZvciBpIGluIGlkeDoKICAgICAgICBmID0gY3YyLnJlc2l6ZShmcmFtZXNbaV0sICg0ODAsIGludCg0ODAgKiBmcmFtZXNbaV0uc2hhcGVbMF0gLyBmcmFtZXNbaV0uc2hhcGVbMV0pKSkKICAgICAgICB0ID0gaSAvIChmcHMgb3IgMjQpOyB4ID0gZGV0LmdldChyb3VuZCh0LCAxKSkgb3IgbWluKHJlcy5nZXQoImRldGFsbGUiLCBbe31dKSBvciBbe31dLCBrZXk9bGFtYmRhIHI6IGFicyhyLmdldCgidCIsIDApIC0gdCkpCiAgICAgICAgbWFsbyA9IGJvb2woeC5nZXQoInByb2JsZW1hcyIpKQogICAgICAgIGN2Mi5yZWN0YW5nbGUoZiwgKDAsIDApLCAoZi5zaGFwZVsxXSAtIDEsIGYuc2hhcGVbMF0gLSAxKSwgKDAsIDAsIDI1NSkgaWYgbWFsbyBlbHNlICgwLCAxNzAsIDApLCA2KQogICAgICAgIHR4dCA9IGYie3Q6LjFmfXMgaWQge3guZ2V0KCdpZGVudGlkYWQnLCAnLScpfSIgKyAoIiB8ICIgKyAiLCAiLmpvaW4oeFsicHJvYmxlbWFzIl0pWzo0OF0gaWYgbWFsbyBlbHNlICIiKQogICAgICAgIGN2Mi5wdXRUZXh0KGYsIHR4dCwgKDEwLCAyNiksIGN2Mi5GT05UX0hFUlNIRVlfU0lNUExFWCwgMC41NSwgKDI1NSwgMjU1LCAyNTUpLCAyLCBjdjIuTElORV9BQSkKICAgICAgICB0aWxlcy5hcHBlbmQoZikKICAgIGZpbGFzID0gW25wLmhzdGFjayh0aWxlc1tpOmkgKyA0XSkgZm9yIGkgaW4gcmFuZ2UoMCwgbGVuKHRpbGVzKSwgNCldCiAgICBjdjIuaW13cml0ZShzYWxpZGEsIG5wLnZzdGFjayhmaWxhcyksIFtjdjIuSU1XUklURV9KUEVHX1FVQUxJVFksIDg4XSkKCgpkZWYgaHVlbGxhX2RlKG1lZCwgaW1nX3BhdGgpOgogICAgaW1nID0gY3YyLmltcmVhZChpbWdfcGF0aCk7IGhzID0gbWVkLmh1ZWxsYXMoaW1nKQogICAgcmV0dXJuIG1heChocywga2V5PWxhbWJkYSB4OiB4WzBdWzJdICogeFswXVszXSlbMV0gaWYgaHMgZWxzZSBOb25lCgoKZGVmIGVsZWdpcihjZmcpOgogICAgIiIiRWxpZ2UgbGEgbWVqb3IgY2FuZGlkYXRhOiB1bmEgc29sYSBjYXJhLCBncmFuZGUsIGRlIGZyZW50ZSwgc2ltw6l0cmljYSwgbsOtdGlkYSB5IChzaSBoYXkgaWRlbnRpZGFkIGZpamEpIGxhIG3DoXMgcGFyZWNpZGEuIiIiCiAgICBtZWQgPSBNZWRpZG9yKGNmZ1sibW9kZWxvcyJdKTsgaWRlbnQgPSBodWVsbGFfZGUobWVkLCBjZmdbImlkZW50aWRhZCJdKSBpZiBjZmcuZ2V0KCJpZGVudGlkYWQiKSBlbHNlIE5vbmUKICAgIGZtID0gbWVkLm1wLnNvbHV0aW9ucy5mYWNlX21lc2guRmFjZU1lc2goc3RhdGljX2ltYWdlX21vZGU9VHJ1ZSwgbWF4X251bV9mYWNlcz0zLCByZWZpbmVfbGFuZG1hcmtzPVRydWUpCiAgICBwdW50b3MgPSBbXQogICAgZm9yIHAgaW4gY2ZnWyJjYW5kaWRhdGFzIl06CiAgICAgICAgaW1nID0gY3YyLmltcmVhZChwKTsgaCwgdyA9IGltZy5zaGFwZVs6Ml07IGhzID0gbWVkLmh1ZWxsYXMoaW1nKQogICAgICAgIHIgPSBmbS5wcm9jZXNzKGN2Mi5jdnRDb2xvcihpbWcsIGN2Mi5DT0xPUl9CR1IyUkdCKSkKICAgICAgICBpbmZvID0geyJpbWciOiBwLCAiY2FyYXMiOiBsZW4oaHMpfQogICAgICAgIGlmIGxlbihocykgIT0gMSBvciBub3Qgci5tdWx0aV9mYWNlX2xhbmRtYXJrczoKICAgICAgICAgICAgaW5mb1sicHVudGFqZSJdID0gLTk7IHB1bnRvcy5hcHBlbmQoaW5mbyk7IGNvbnRpbnVlCiAgICAgICAgYm94LCBodSA9IGhzWzBdCiAgICAgICAgcHRzID0gbnAuYXJyYXkoWyhxLnggKiB3LCBxLnkgKiBoKSBmb3IgcSBpbiByLm11bHRpX2ZhY2VfbGFuZG1hcmtzWzBdLmxhbmRtYXJrXSwgbnAuZmxvYXQzMikKICAgICAgICBnID0gZ2VvbWV0cmlhKHB0cykKICAgICAgICBpbmZvLnVwZGF0ZSh7InRhbSI6IHJvdW5kKGZsb2F0KGJveFszXSkgLyBoLCAzKSwgImFzaW1fb2pvcyI6IHJvdW5kKGdbImFzaW1fb2pvcyJdLCAzKSwgImdpcm8iOiByb3VuZChnWyJnaXJvIl0sIDMpLCAibml0aWRleiI6IHJvdW5kKG5pdGlkZXooaW1nLCBib3gpLCAxKX0pCiAgICAgICAgaW5mb1sicGFyZWNpZG8iXSA9IHJvdW5kKG1lZC5zaW0oaHUsIGlkZW50KSwgMykgaWYgaWRlbnQgaXMgbm90IE5vbmUgZWxzZSBOb25lCiAgICAgICAgZGVmZWN0b3MgPSAoZ1siYXNpbV9vam9zIl0gPiAwLjI1KSArIChnWyJsYWJpb3NfY3J1emFkb3MiXSA+IDAuMDIpICsgKG5vdCAwLjMwIDw9IGdbImJvY2FfYW5jaG8iXSA8PSAxLjMwKQogICAgICAgIGluZm9bInB1bnRhamUiXSA9IHJvdW5kKChpbmZvWyJwYXJlY2lkbyJdIGlmIGlkZW50IGlzIG5vdCBOb25lIGVsc2UgMCkgKiAzIC0gZGVmZWN0b3MgLSBhYnMoZ1siZ2lybyJdKSArIG1pbihpbmZvWyJuaXRpZGV6Il0sIDQwMCkgLyA0MDAsIDMpCiAgICAgICAgcHVudG9zLmFwcGVuZChpbmZvKQogICAgZm0uY2xvc2UoKQogICAgcHVudG9zLnNvcnQoa2V5PWxhbWJkYSB4OiAteFsicHVudGFqZSJdKQogICAganNvbi5kdW1wKHB1bnRvcywgb3BlbihjZmdbInNhbGlkYSJdLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0xKQogICAgcHJpbnQoanNvbi5kdW1wcyhwdW50b3NbMF0sIGVuc3VyZV9hc2NpaT1GYWxzZSkpCgoKZGVmIG1haW4oY2ZnKToKICAgIG1lZCA9IE1lZGlkb3IoY2ZnWyJtb2RlbG9zIl0pCiAgICByZWZzID0ge2M6IGggZm9yIGMsIHAgaW4gY2ZnLmdldCgicmVmcyIsIHt9KS5pdGVtcygpIGlmIChoIDo9IGh1ZWxsYV9kZShtZWQsIHApKSBpcyBub3QgTm9uZX0KICAgIG91dCA9IHt9CiAgICBmb3IgcGwgaW4gY2ZnWyJwbGFub3MiXToKICAgICAgICByID0gYW5hbGl6YXIobWVkLCBwbFsidmlkZW8iXSwgcmVmcywgcGwuZ2V0KCJlc3BlcmFkb3MiKSBvciBbXSwgZXNjZW5hPXBsLmdldCgiZXNjZW5hIiksIHVuaWNhPWJvb2wocGwuZ2V0KCJ1bmljYSIpKSkKICAgICAgICBpZiBwbC5nZXQoImhvamEiKTogaG9qYV9jb250YWN0byhwbFsidmlkZW8iXSwgciwgcGxbImhvamEiXSkKICAgICAgICBvdXRbcGwuZ2V0KCJpZCIpIG9yIHJbInZpZGVvIl1dID0gcgogICAgICAgIHByaW50KGpzb24uZHVtcHMoe2s6IHJba10gZm9yIGsgaW4gKCJ2aWRlbyIsICJhcHJvYmFkbyIsICJpZGVudGlkYWRfbWVkaWEiLCAiaWRlbnRpZGFkX2VzY2VuYV9tZWRpYSIsICJjdWFkcm9zX2Nvbl9kZWZlY3RvIiwgInByb2JsZW1hcyIpfSwgZW5zdXJlX2FzY2lpPUZhbHNlKSwgZmx1c2g9VHJ1ZSkKICAgICMgQ2FsaWJyYWNpw7NuIGNvbiBkYXRvcyByZWFsZXM6IHBhcmVjaWRvIGVudHJlIHBsYW5vcyB5IHJlZmVyZW5jaWFzIChtaXNtYSBwZXJzb25hIHZzLiBwZXJzb25hcyBkaXN0aW50YXMpCiAgICBub21icmVzID0gW24gZm9yIG4gaW4gb3V0IGlmIG91dFtuXS5nZXQoIl9odWVsbGEiKV0KICAgIEggPSB7bjogbnAuYXJyYXkob3V0W25dWyJfaHVlbGxhIl0sIG5wLmZsb2F0MzIpIGZvciBuIGluIG5vbWJyZXN9CiAgICBmb3IgaywgaHUgaW4gcmVmcy5pdGVtcygpOiBIW2YicmVmOntrfSJdID0gaHUKICAgIGNsYXZlcyA9IGxpc3QoSCkKICAgIGNhbCA9IHthOiB7Yjogcm91bmQobWVkLnNpbShIW2FdLCBIW2JdKSwgMykgZm9yIGIgaW4gY2xhdmVzIGlmIGIgIT0gYX0gZm9yIGEgaW4gY2xhdmVzfQogICAgZm9yIG4gaW4gb3V0OiBvdXRbbl0ucG9wKCJfaHVlbGxhIiwgTm9uZSkKICAgIGpzb24uZHVtcChvdXQsIG9wZW4oY2ZnWyJzYWxpZGEiXSwgInciLCBlbmNvZGluZz0idXRmLTgiKSwgZW5zdXJlX2FzY2lpPUZhbHNlLCBpbmRlbnQ9MSkKICAgIGpzb24uZHVtcChjYWwsIG9wZW4oY2ZnWyJzYWxpZGEiXS5yZXBsYWNlKCIuanNvbiIsICJfY2FsaWJyYWNpb24uanNvbiIpLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpLCBlbnN1cmVfYXNjaWk9RmFsc2UsIGluZGVudD0xKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBpZiBzeXMuYXJndlsxXSA9PSAiLS1lbGVnaXIiOiBlbGVnaXIoanNvbi5sb2FkKG9wZW4oc3lzLmFyZ3ZbMl0sIGVuY29kaW5nPSJ1dGYtOCIpKSkKICAgIGVsc2U6IG1haW4oanNvbi5sb2FkKG9wZW4oc3lzLmFyZ3ZbMV0sIGVuY29kaW5nPSJ1dGYtOCIpKSkK"}.items():
    _pl.Path("/tmp", _n).write_bytes(_b64.b64decode(_c))
r = sh("uv venv -q -p 3.10 /tmp/mt && uv pip install -q -p /tmp/mt/bin/python torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121 && "
       "uv pip install -q -p /tmp/mt/bin/python diffusers==0.30.2 transformers==4.39.2 accelerate==0.28.0 huggingface_hub==0.24.6 librosa==0.10.2.post1 einops==0.8.1 "
       "opencv-python-headless==4.9.0.80 mediapipe==0.10.14 numpy==1.26.4 soundfile", check=False)
if r.returncode == 0:
    r = sh(f"rm -rf {MT_REPO} && git clone -q https://github.com/TMElyralab/MuseTalk.git {MT_REPO} && cd {MT_REPO} && git checkout -q 0a89dec45a0192b824e3cf4daf96c239440c5ed8", check=False)
if r.returncode == 0:
    _dl = (f"from huggingface_hub import hf_hub_download as d; M='{MTM}'\n"
           "for f in ['musetalkV15/musetalk.json','musetalkV15/unet.pth']: d('TMElyralab/MuseTalk', f, local_dir=M)\n"
           "for f in ['config.json','diffusion_pytorch_model.bin']: d('stabilityai/sd-vae-ft-mse', f, local_dir=M+'/sd-vae')\n"
           "for f in ['config.json','pytorch_model.bin','preprocessor_config.json']: d('openai/whisper-tiny', f, local_dir=M+'/whisper')\n")
    _pl.Path("/tmp/mt_pesos.py").write_text(_dl)
    r = sh(f"{MT_PY} /tmp/mt_pesos.py", check=False)
if r.returncode == 0:
    for _u in ("face_detection_yunet/face_detection_yunet_2023mar.onnx", "face_recognition_sface/face_recognition_sface_2021dec.onnx"):
        sh(f"curl -sSL -o {MTM}/{_u.split('/')[1]} https://github.com/opencv/opencv_zoo/raw/main/models/{_u}", check=False)
    _ok = all(_pl.Path(MTM, f).exists() and _pl.Path(MTM, f).stat().st_size > 100_000 for f in ("musetalkV15/unet.pth", "sd-vae/diffusion_pytorch_model.bin", "whisper/pytorch_model.bin",
                                                                                                   "face_detection_yunet_2023mar.onnx", "face_recognition_sface_2021dec.onnx"))
    r = subprocess.CompletedProcess("", 0 if _ok else 1, "", "faltan pesos")
MT_OK = r.returncode == 0
print("✅ Labios (MuseTalk 1.5) y control de rostros listos" if MT_OK else "⚠️ Labios/control de rostros NO disponibles: " + (r.stderr or "")[-400:])
try:
    from diffusers import WanPipeline  # noqa: F401
except Exception:
    sh(f"{sys.executable} -m pip install -q -U git+https://github.com/huggingface/diffusers")
import diffusers, torch
print("✅ diffusers", diffusers.__version__, "· torch", torch.__version__)

## 3 · Producción
- **MODO = "paquete"** (recomendado): sube el `yoha-produccion-….json` exportado desde Yoha Video. Respeta reparto, escenas, planos, actuación, voces, idiomas y solo regenera lo modificado.
- **MODO = "instruccion"**: escribe una sola instrucción y el guionista IA arma todo aquí.
- **AVISO_NTFY**: tema de la app gratuita **ntfy** (Yoha Video → Configuración) para recibir en el iPhone «🎬 Yoha Video: tu producción está lista para revisar».

In [ ]:
MODO = "paquete"            #@param ["paquete", "instruccion"]
INSTRUCCION = "Créame una novelita dominicana de dos minutos, con tres personajes, romance, traición y un final que motive a ver el próximo capítulo"  #@param {type:"string"}
ESCRIBIR_GUION_CON_IA = True #@param {type:"boolean"}
MODELO_VIDEO = "auto"       #@param ["auto", "wan2.1-1.3b", "wan2.2-5b"]
CALIDAD = "cine"            #@param ["borrador", "normal", "alta", "cine"]
COMPARAR_MODELOS = False    #@param {type:"boolean"}
VOZ_MOTOR = "chatterbox"    #@param ["chatterbox", "kokoro"]
MUSICA = True               #@param {type:"boolean"}
MAX_PLANOS_ESTA_SESION = 24 #@param {type:"integer"}
VOCES = True                #@param {type:"boolean"}
SUBTITULOS_WHISPER = True   #@param {type:"boolean"}
LABIOS_SINCRONIZADOS = True #@param {type:"boolean"}
CONTROL_ROSTROS = True      #@param {type:"boolean"}
REINTENTOS_ROSTRO = 2       #@param {type:"integer"}
CANDIDATAS_REF = 4          #@param {type:"integer"}
REVISAR_PARECIDO = True     #@param {type:"boolean"}
UMBRAL_PARECIDO = 0.72      #@param {type:"number"}
ESCALAR_A = "ninguno"       #@param ["ninguno", "1080p", "4k"]
AVISO_NTFY = ""             #@param {type:"string"}
LIMITE_HORAS = 11.0         #@param {type:"number"}

# Modo automático: la Productora de Yoha Video (GitHub Actions) antepone una celda con YOHA_AUTO = {...}
YOHA_AUTO = globals().get("YOHA_AUTO")
if YOHA_AUTO:
    MODO = "paquete"
    globals().update(YOHA_AUTO.get("params", {}))

import json, os, glob, pathlib, re, requests, hashlib
OUT = pathlib.Path("/kaggle/working/yoha_clips" if os.path.isdir("/kaggle/working") else "/content/yoha_clips"); OUT.mkdir(parents=True, exist_ok=True)

def avisar(kind, detalle=""):
    msgs = {"ready": "🎬 Yoha Video: tu producción está lista para revisar.", "error": "⚠️ Yoha Video: ocurrió un error crítico en tu producción.", "stopped": "⏸️ Yoha Video: la producción se detuvo antes de terminar."}
    if not re.fullmatch(r"[A-Za-z0-9_-]{12,64}", AVISO_NTFY or ""): return
    try: requests.post(f"https://ntfy.sh/{AVISO_NTFY}", data=(msgs[kind] + ("\n" + detalle if detalle else "")).encode(), headers={"Title": "Yoha Video", "Priority": "default" if kind == "ready" else "high"}, timeout=10)
    except Exception as e: print("Aviso no enviado:", e)

STYLE = "cinematic film still, natural skin texture, soft key light, shallow depth of field, 35mm, color graded, telenovela production quality"
EMO = {"neutral": "calm neutral expression", "feliz": "smiling warmly, happy", "risa": "laughing naturally", "triste": "sad, glossy eyes", "llanto": "crying, tears running down the cheeks", "enfado": "angry, furrowed brow, clenched jaw", "sorpresa": "surprised, eyes wide open", "miedo": "afraid, tense", "amor": "loving gaze, soft smile", "celos": "jealous, narrowed eyes", "determinacion": "determined, steady gaze"}
GEST = {"ninguno": "", "asiente": "nods slowly", "niega": "shakes head", "manos": "gestures with both hands while talking", "abrazo": "hugs the other person", "señala": "points at someone", "cubre_rostro": "covers face with hands", "toma_mano": "holds the other person's hand", "brazos_cruzados": "crosses arms"}
MOVE = {"quieto": "", "camina": "walks slowly toward the camera", "entra": "enters the room through the door", "se_va": "turns and walks away", "se_sienta": "sits down", "se_levanta": "stands up abruptly", "gira": "turns to face the other character", "corre": "runs"}

if YOHA_AUTO:
    PKG = YOHA_AUTO["pkg"]
    assert PKG.get("format") == "yoha-production", "No es un paquete de Yoha Video."
    # Reanudar: clips y guion de la sesión anterior (salida del kernel previo, añadida como entrada)
    import shutil as _sh
    prev = [pathlib.Path(f) for f in glob.glob("/kaggle/input/**/yoha_clips/*", recursive=True)]
    for f in prev:
        if f.suffix == ".mp4" and not (OUT / f.name).exists(): _sh.copy(f, OUT / f.name)
    for f in glob.glob("/kaggle/input/**/yoha_clips_*/*.mp4", recursive=True):   # planos de la comparación (Wan 2.2)
        dv = OUT.parent / pathlib.Path(f).parent.name; dv.mkdir(exist_ok=True)
        if not (dv / pathlib.Path(f).name).exists(): _sh.copy(f, dv / pathlib.Path(f).name)
    for f in glob.glob("/kaggle/input/**/yoha_clips/referencias/**/*.*", recursive=True):   # identidades fijas y primeros cuadros
        dst = OUT / "referencias" / f.split("/yoha_clips/referencias/", 1)[1]
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists(): _sh.copy(f, dst)
    for f in prev:
        if f.name in ("control_rostros.json", "labios.json", "labios_informe.json", "cortes.json") and not (OUT / f.name).exists(): _sh.copy(f, OUT / f.name)
    for sub in ("_sin_labios", "_intentos", "_completos"):
        for f in glob.glob(f"/kaggle/input/**/{sub}/*.mp4", recursive=True):
            d = OUT.parent / sub; d.mkdir(exist_ok=True)
            if not (d / pathlib.Path(f).name).exists(): _sh.copy(f, d / pathlib.Path(f).name)
    g = next((f for f in prev if f.name == "guion_ia.json"), None)
    if g and not any(x.get("dialogue") for x in PKG["scenes"]):
        G = json.load(open(g, encoding="utf-8"))
        PKG["title"], PKG["synopsis"] = G.get("title", PKG.get("title")), G.get("synopsis", PKG.get("synopsis", ""))
        if G.get("characters"): PKG["characters"] = G["characters"]
        PKG["scenes"] = G["scenes"]; _sh.copy(g, OUT / "guion_ia.json")
    print(f"Modo automático · {len([f for f in prev if f.suffix == '.mp4'])} clips de la sesión anterior")
elif MODO == "paquete":
    files = glob.glob("/kaggle/input/**/yoha-*.json", recursive=True)
    if not files:
        try:
            from google.colab import files as gf  # type: ignore
            print("Sube tu yoha-produccion-….json"); files = list(gf.upload().keys())
        except ImportError:
            raise SystemExit("Kaggle: Add Input → Upload → sube el JSON de Yoha Video y vuelve a ejecutar.")
    PKG = json.load(open(files[0], encoding="utf-8"))
    assert PKG.get("format") == "yoha-production", "No es un paquete de Yoha Video."
else:
    PKG = {"format": "yoha-production", "version": 2, "title": "Producción YDC", "instruction": INSTRUCCION, "styleText": STYLE, "aspect": "9:16" if re.search(r"short|vertical|tiktok|reel", INSTRUCCION, re.I) else "16:9", "languages": ["es"] + (["en"] if re.search(r"ingl|english|doblaj", INSTRUCCION, re.I) else []), "allowStrongLanguage": False, "characters": [], "scenes": [], "brief": None}
STYLE = PKG.get("styleText") or STYLE
print(f"«{PKG.get('title')}» · {len(PKG['scenes'])} escenas · {len(PKG['characters'])} personajes · idiomas {PKG.get('languages')} · {'solo escenas modificadas' if PKG.get('onlyChanged') else 'completa'}")

## 4 · Director IA: guion completo con Qwen2.5-7B (gratis, lenguaje limpio por defecto)

In [ ]:
BAD = ["mierda","carajo","coño","joder","puta","puto","pendejo","pendeja","cabron","cabrón","chingar","verga","huevon","maricon","gilipollas","culero","mamaguevo","hijo de puta","comemierda","singar","fuck","shit","bitch","bastard","asshole","motherfucker","cunt","whore","slut"]
def groserias(t):
    t = (t or "").lower(); return [w for w in BAD if re.search(rf"(^|\W){re.escape(w)}(?=$|\W)", t)]

def build_brief():
    if PKG.get("brief"): return PKG["brief"]
    return f"""Eres guionista profesional para el canal YDC Entertainment Virales. Instrucción: "{PKG['instruction']}".
Crea una historia 100% original en español, lenguaje limpio (sin malas palabras). Define 2 a 6 personajes con id cortos (p1, p2…), escenas de 15-40 s divididas en planos de ~5 s, diálogos breves.
Responde SOLO JSON: {{"title":"","synopsis":"","characters":[{{"id":"p1","name":"","role":"","appearance":"descripción física en inglés","wardrobe":"en inglés","voice":"ef_dora|em_alex|em_santa","voiceEn":"af_heart|am_michael"}}],"scenes":[{{"id":"e1","title":"","location":"","time":"día|atardecer|noche","description":"","descriptionEn":"visual description in English","camera":"","characters":["p1"],"shots":3,"narration":"","dialogue":[{{"character":"p1","text":"","emotion":"neutral|feliz|risa|triste|llanto|enfado|sorpresa|miedo|amor|celos|determinacion"}}],"dialogueEn":[{{"character":"p1","text":""}}],"musicMood":"","sfx":""}}]}}"""

def write_script():
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    mid = "Qwen/Qwen2.5-7B-Instruct"
    tok = AutoTokenizer.from_pretrained(mid)
    q = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    llm = AutoModelForCausalLM.from_pretrained(mid, quantization_config=q, device_map="auto")
    brief = build_brief()
    for intento in range(2):
        msgs = [{"role": "system", "content": "Respondes solo con JSON válido. Historias originales, sin lenguaje vulgar salvo autorización."}, {"role": "user", "content": brief}]
        enc = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True)
        enc = {k: v.to(llm.device) for k, v in enc.items()}
        out = llm.generate(**enc, max_new_tokens=6000, temperature=0.8, top_p=0.9, do_sample=True)
        text = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
        m = re.search(r"\{[\s\S]*\}", text)
        try:
            data = json.loads(m.group(0))
        except Exception:
            print("JSON inválido, reintentando…"); continue
        malas = groserias(json.dumps(data, ensure_ascii=False))
        if malas and not PKG.get("allowStrongLanguage"):
            print("Lenguaje no permitido:", malas, "→ reescribiendo"); brief += "\nIMPORTANTE: elimina cualquier mala palabra."; continue
        break
    del llm; torch.cuda.empty_cache()
    return data

if ESCRIBIR_GUION_CON_IA and (MODO == "instruccion" or not any(s.get("dialogue") for s in PKG["scenes"])):
    if not torch.cuda.is_available(): raise SystemExit("🚫 BLOQUEADO: el guionista necesita GPU.")
    data = write_script()
    if MODO == "instruccion":
        PKG["title"], PKG["synopsis"] = data.get("title", PKG["title"]), data.get("synopsis", "")
        PKG["characters"] = [{**c, "seed": 1000 + i * 101, "references": []} for i, c in enumerate(data.get("characters", []))]
        PKG["scenes"] = [{**s, "n": i + 1, "seed": 2000 + i * 37, "seconds": int(s.get("shots", 3)) * 5, "acting": {}} for i, s in enumerate(data.get("scenes", []))]
    else:
        by_id = {s["id"]: s for s in data.get("scenes", [])}
        for s in PKG["scenes"]:
            n = by_id.get(s["id"])
            if n: s.update({k: n[k] for k in ("title", "description", "descriptionEn", "narration", "dialogue", "dialogueEn", "musicMood", "sfx") if k in n})
    json.dump({"title": PKG["title"], "synopsis": PKG.get("synopsis", ""), "characters": PKG["characters"] if MODO == "instruccion" else None, "scenes": PKG["scenes"]}, open(OUT / "guion_ia.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print("✅ Guion escrito. guion_ia.json se incluye en el .zip (impórtalo en Yoha Video → producción → «Importar guion»).")
else:
    print("Se usa el guion del paquete.")

# Dirección automática: texto final de cada plano con apariencia fija + actuación + cámara
CH = {c["id"]: c for c in PKG["characters"]}
ENCUADRE = {"primer plano": "cinematic close-up shot (face and shoulders filling the frame, shallow depth of field)",
            "plano detalle": "extreme close-up of the face", "plano medio": "medium shot", "plano general": "wide establishing shot"}
def plan(s, k):
    pl = s.get("planos") or []
    d = dict(pl[k]) if k < len(pl) else {}
    if str(k) in (s.get("rehacer") or {}): d["rehacer"] = int(s["rehacer"][str(k)])   # rechazo de un plano en escenas sin dirección por planos
    return d
def planificar(s):
    """Dirección por planos (igual que la app): plano general para situar, primer plano de quien habla y reacción de quien escucha."""
    dl = s.get("dialogue") or []
    lineas = [(j, d) for j, d in enumerate(dl) if d.get("text") and d.get("character") in CH]
    if not lineas: return None
    pl = []
    if len(s.get("characters") or []) >= 2 or s.get("narration"): pl.append({"encuadre": "plano general", "foco": None, "lineas": []})
    for j, d in lineas:
        prev = pl[-1] if pl else None
        txt = " ".join(dl[i]["text"] for i in (prev or {}).get("lineas", []))
        if prev and prev.get("foco") == d["character"] and prev.get("encuadre") == "primer plano" and len(txt + d["text"]) <= 90: prev["lineas"].append(j)
        else: pl.append({"encuadre": "primer plano", "foco": d["character"], "lineas": [j], "emocion": d.get("emotion")})
    ultimo = lineas[-1][1]["character"]
    oyente = next((c for c in s.get("characters") or [] if c != ultimo and any(d["character"] == c for _, d in lineas)), None) or next((c for c in s.get("characters") or [] if c != ultimo and c in CH), None)
    if oyente: pl.append({"encuadre": "primer plano", "foco": oyente, "lineas": [], "accion": "listening silently, emotional reaction, lips closed"})
    return pl
def es_primer_plano(s, k):
    p = plan(s, k); return bool(p.get("foco") in CH and p.get("encuadre") in ("primer plano", "plano detalle"))
def habla_en(s, k, cid):
    dl = s.get("dialogue", []); return any(j < len(dl) and dl[j].get("character") == cid for j in plan(s, k).get("lineas", []))
def prompt_for(s, k):
    light = {"noche": "night", "atardecer": "golden hour sunset light"}.get(s.get("time"), "daylight")
    if es_primer_plano(s, k):
        p = plan(s, k); c = CH[p["foco"]]; a = (s.get("acting") or {}).get(p["foco"], {})
        emo = p.get("emocion") or a.get("emotion") or next((d.get("emotion") for d in s.get("dialogue", []) if d.get("character") == p["foco"] and d.get("emotion")), "neutral")
        accion = p.get("accion") or ("speaking to someone just off camera, natural lip and jaw movement, expressive eyes" if habla_en(s, k, p["foco"])
                                      else "listening silently with the mouth closed, subtle natural reactions")
        return ". ".join(x for x in [f"{ENCUADRE[p['encuadre']]}: {c['name']}, {c['appearance']}" + (f", wearing {c['wardrobe']}" if c.get("wardrobe") else ""),
                                      EMO.get(emo, ""), accion, f"background softly out of focus: {s.get('location', '')}, {light}",
                                      "only this one person in the frame, nobody else visible at the edges, steady face with consistent features, natural skin texture", STYLE] if x) + "."
    people = []
    for cid in s.get("characters", []):
        c = CH.get(cid);
        if not c: continue
        a = (s.get("acting") or {}).get(cid, {})
        emo = a.get("emotion") or next((d.get("emotion") for d in s.get("dialogue", []) if d.get("character") == cid and d.get("emotion")), "neutral")
        cue = ", ".join(x for x in [EMO.get(emo, ""), GEST.get(a.get("gesture", ""), ""), MOVE.get(a.get("move", ""), "")] if x)
        people.append(f"{c['name']}, {c['appearance']}" + (f", wearing {c['wardrobe']}" if c.get("wardrobe") else "") + (f" — {cue}" if cue else ""))
    talk = " The characters talk to each other, lips moving naturally." if s.get("dialogue") else ""
    return ". ".join(x for x in [f"{s.get('location', '')}, {light}", s.get("descriptionEn") or s.get("description", ""), people and "Characters: " + "; ".join(people), f"Camera: {s.get('camera', 'medium shot')}" + (", reverse angle" if k % 2 else ""), STYLE] if x) + "." + talk
for s in PKG["scenes"]:
    if not s.get("planos") and s.get("dialogue"):   # el guion llegó sin dirección por planos (p. ej. lo escribió la IA aquí): se planifica
        pl = planificar(s)
        if pl: s["planos"], s["shots"] = pl, len(pl)
    s["hash"] = s.get("hash") or hashlib.sha1(json.dumps([s.get("descriptionEn") or s.get("description"), s.get("characters"), s.get("acting"), s.get("seed"), s.get("shots")]
                                                         + ([s["planos"]] if s.get("planos") else []), sort_keys=True).encode()).hexdigest()[:8]
def prompt_identidad(c):
    return (f"Photorealistic cinematic portrait of {c['name']}, {c['appearance']}" + (f", wearing {c['wardrobe']}" if c.get("wardrobe") else "") +
            ", neutral relaxed expression, mouth closed, facing the camera, soft even light, plain softly blurred background, sharp detailed eyes, natural skin texture, 85mm lens, " + STYLE)
json.dump({"scenes": [{"n": s["n"], "hash": s["hash"], "shots": s.get("shots"), "planos": s.get("planos"), "dialogue": s.get("dialogue")} for s in PKG["scenes"]]},
          open(OUT / "plan_final.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)   # la Productora controla la calidad con ESTE plan
print("Ejemplo de dirección:\n", prompt_for(PKG["scenes"][0], 0)[:600])

## 5 · Modelos de video y dirección de cada plano (texto → embeddings, una sola vez)

In [ ]:
import gc, time
if not torch.cuda.is_available():
    avisar("stopped", "No hay GPU disponible."); raise SystemExit("🚫 BLOQUEADO: no hay GPU. No se generan videos simulados.")
NGPU = torch.cuda.device_count()
vram = min(torch.cuda.get_device_properties(i).total_memory for i in range(NGPU)) / 1024**3
cap = torch.cuda.get_device_capability(0)
DTYPE = "bfloat16" if cap[0] >= 8 else "float16"   # T4 (7.5) no tiene bf16 nativo
# repo, fps, cuadros, (ancho, alto) 16:9, flow_shift (oficial: 3.0 a 480p, 5.0 a 720p)
SPECS = {"wan2.1-1.3b": ("Wan-AI/Wan2.1-T2V-1.3B-Diffusers", 16, 81, (832, 480), 3.0),
         "wan2.2-5b": ("Wan-AI/Wan2.2-TI2V-5B-Diffusers", 24, 121, (1280, 704), 5.0)}
MODELOS = ["wan2.1-1.3b", "wan2.2-5b"] if COMPARAR_MODELOS else [MODELO_VIDEO if MODELO_VIDEO != "auto" else ("wan2.2-5b" if vram >= 22 else "wan2.1-1.3b")]
choice = MODELOS[0]
STEPS = {"borrador": 20, "normal": 30, "alta": 40, "cine": 50}[CALIDAD]
GUIA = 5.0
def dims(m):
    W0, H0 = SPECS[m][3]
    return {"16:9": (W0, H0), "9:16": (H0, W0), "1:1": (H0, H0), "4:5": (H0, int(H0 * 1.25) // 32 * 32)}.get(PKG.get("aspect", "16:9"), (W0, H0))
W, H = dims(choice); FPS = SPECS[choice][1]
NEG = ("色调艳丽，过曝，静态，细节模糊不清，字幕，风格，作品，画作，画面，静止，整体发灰，最差质量，低质量，JPEG压缩残留，丑陋的，残缺的，多余的手指，画得不好的手部，画得不好的脸部，畸形的，毁容的，形态畸形的肢体，手指融合，静止不动的画面，杂乱的背景，三条腿，背景人很多，倒着走, "
       "blurry, low quality, distorted face, deformed hands, extra fingers, text, subtitles, watermark, logo, flicker, bad anatomy, cartoon, plastic skin, extra person, second face, partial face at the edge of the frame, asymmetric eyes, deformed mouth")
DIRS = {m: (OUT if i == 0 else OUT.parent / f"yoha_clips_{m.replace('.', '').replace('-', '_')}") for i, m in enumerate(MODELOS)}
for d in DIRS.values(): d.mkdir(exist_ok=True)
EMB = pathlib.Path("/tmp/yoha_emb"); EMB.mkdir(exist_ok=True)   # fuera de la salida de Kaggle
TAREAS = {m: [] for m in MODELOS}
I2V = {"wan2.2-5b"}                 # Wan 2.2 TI2V anima una imagen fija: el rostro del primer cuadro se conserva en todo el plano
REF = OUT / "referencias"; (REF / "cand").mkdir(parents=True, exist_ok=True)
IMG_TAREAS = {m: [] for m in MODELOS}
TODAS = {}                           # todos los planos (también los ya hechos), para poder regenerar solo los defectuosos
ELEGIR = []                          # (destino, candidatas, identidad de referencia o None)
def _cand(m, key, prompt, seed0):
    outs = []
    for i in range(max(1, int(CANDIDATAS_REF))):
        f = REF / "cand" / f"{key}_{i}.png"; outs.append(str(f))
        if not f.exists():
            IMG_TAREAS[m].append({"tipo": "imagen", "out": str(f), "emb": str(EMB / f"{m}_img_{key}.pt"), "seed": seed0 + i * 7919, "prompt": prompt})
    return outs
focos = sorted({plan(s, k)["foco"] for s in PKG["scenes"] for k in range(int(s.get("shots", 1))) if es_primer_plano(s, k)})
for cid, c in CH.items():   # referencia OFICIAL aprobada (personajes/<id>.jpg en el repositorio): manda sobre cualquier otra
    if c.get("referencia"):
        import io as _io, base64 as _b64r
        from PIL import Image as _Im
        _Im.open(_io.BytesIO(_b64r.b64decode(c["referencia"]))).convert("RGB").save(REF / f"{cid}.png"); print(f"🪪 referencia oficial de {c.get('name', cid)}")
for cid in focos:   # 1) identidad fija de cada personaje (retrato neutro) — se reutiliza en todas las escenas
    if not (REF / f"{cid}.png").exists():
        ELEGIR.append((REF / f"{cid}.png", _cand(choice, f"id_{cid}", prompt_identidad(CH[cid]), int(CH[cid].get("seed", 1000))), None))
for s in PKG["scenes"]:
    for k in range(int(s.get("shots", 1))):
        cid = plan(s, k).get("foco") if es_primer_plano(s, k) else None
        for m in MODELOS:
            f = DIRS[m] / f"e{int(s['n']):03d}_p{k+1:02d}_{s['hash']}.mp4"
            seed = int(CH[cid].get("seed", 1000)) * 3 + int(s["n"]) * 31 + k if cid else int(s.get("seed", 1)) + k * 101
            t = {"out": str(f), "emb": str(EMB / f"{m}_{f.stem}.pt"), "seed": seed, "prompt": prompt_for(s, k)}
            if cid and m in I2V:   # 2) primer cuadro de la escena para ese personaje, elegido por parecido con su identidad fija
                key = f"e{int(s['n']):03d}_{s['hash']}_{cid}"; t["imagen"] = str(REF / f"{key}.png")
                if not (REF / f"{key}.png").exists() and not any(str(d) == t["imagen"] for d, _, _ in ELEGIR):
                    ELEGIR.append((REF / f"{key}.png", _cand(m, key, prompt_for(s, k) + " Still frame, mouth gently closed.", seed), REF / f"{cid}.png"))
            TODAS.setdefault(m, []).append(t)
            if not (f.exists() and f.stat().st_size > 10_000):
                TAREAS[m].append(t)
pend = sum(len(v) for v in TAREAS.values())
print(f"GPU: {NGPU} × {torch.cuda.get_device_name(0)} · {vram:.1f} GB · {DTYPE} · modelos {MODELOS} · {STEPS} pasos · planos pendientes {pend}")
# Texto → embeddings con el codificador umT5 en GPU (una vez); luego se libera y los modelos de video caben sin descargar a CPU.
from diffusers import WanPipeline
for m in MODELOS:
    falta = [t for t in (TODAS.get(m, []) if CONTROL_ROSTROS else TAREAS[m]) + IMG_TAREAS[m] if not os.path.exists(t["emb"])]
    if not falta: continue
    tp = WanPipeline.from_pretrained(SPECS[m][0], transformer=None, vae=None, torch_dtype=getattr(torch, DTYPE)); tp.text_encoder.to("cuda:0")
    with torch.no_grad():
        for t in falta:
            pe, ne = tp.encode_prompt(prompt=t["prompt"], negative_prompt=NEG, do_classifier_free_guidance=True, num_videos_per_prompt=1, max_sequence_length=512, device="cuda:0")
            torch.save({"p": pe.cpu(), "n": ne.cpu()}, t["emb"])
    del tp; gc.collect(); torch.cuda.empty_cache()
print("✅ Dirección de cada plano codificada")

## 6 · Generar planos en paralelo en cada GPU (reanuda; solo regenera escenas modificadas)

In [ ]:
import subprocess, threading
WORKER = OUT.parent / "_worker.py"
WORKER.write_text("""
import sys, os, json, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch
from diffusers import AutoencoderKLWan, WanPipeline, UniPCMultistepScheduler
from diffusers.utils import export_to_video
from PIL import Image
import numpy as np
job = json.load(open(sys.argv[1])); g = job["gpu"]; dev = f"cuda:{g}"; dt = getattr(torch, job["dtype"])
torch.cuda.set_device(g)
vae = AutoencoderKLWan.from_pretrained(job["repo"], subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(job["repo"], vae=vae, text_encoder=None, tokenizer=None, torch_dtype=dt)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=job["shift"])
offload = False
def montar():
    global offload
    if offload:   # pesos por bloques entre CPU y GPU: misma calidad, cabe en 15 GB
        pipe.to("cpu")
        pipe.transformer.enable_group_offload(onload_device=torch.device(dev), offload_device=torch.device("cpu"), offload_type="block_level", num_blocks_per_group=1, use_stream=True)
        pipe.vae.to(dev)
    else: pipe.to(dev)
offload = bool(job.get("offload"))
montar()
try: pipe.vae.enable_tiling()
except Exception: pass
i2v = None
def pipe_i2v():
    global i2v
    if i2v is None:
        from diffusers import WanImageToVideoPipeline
        i2v = WanImageToVideoPipeline(tokenizer=None, text_encoder=None, vae=pipe.vae, scheduler=pipe.scheduler, transformer=pipe.transformer, expand_timesteps=True)
    return i2v
print(json.dumps({"listo": job["modelo"], "gpu": g}), flush=True)
for t in job["tareas"]:
    if os.path.exists(t["out"]): continue
    if time.time() > job["limite"]: print(json.dumps({"limite": True}), flush=True); break
    e = torch.load(t["emb"]); t0 = time.time()
    for intento in range(2):
        try:
            kw = dict(prompt_embeds=e["p"].to(dev, dt), negative_prompt_embeds=e["n"].to(dev, dt), height=job["h"], width=job["w"],
                      num_frames=1 if t.get("tipo") == "imagen" else job["frames"], num_inference_steps=job["steps"], guidance_scale=job["guia"],
                      generator=torch.Generator(dev).manual_seed(t["seed"]))
            if t.get("imagen"):
                fr = pipe_i2v()(image=Image.open(t["imagen"]).convert("RGB").resize((job["w"], job["h"]), Image.LANCZOS), **kw).frames[0]
            else:
                fr = pipe(**kw).frames[0]
            break
        except torch.OutOfMemoryError:
            if offload: raise
            print(json.dumps({"aviso": "memoria insuficiente: se activa descarga a CPU (más lento, misma calidad)"}), flush=True)
            fr = None; torch.cuda.empty_cache(); offload = True; montar()
    brillo = float(np.asarray(fr).mean())
    if t.get("tipo") == "imagen":
        tmp = t["out"] + ".tmp.png"; Image.fromarray((np.asarray(fr)[0] * 255).round().clip(0, 255).astype("uint8")).save(tmp); os.replace(tmp, t["out"])
    else:
        tmp = t["out"] + ".tmp.mp4"; export_to_video(fr, tmp, fps=job["fps"]); os.replace(tmp, t["out"])
    print(json.dumps({"hecho": os.path.basename(t["out"]), "s": round(time.time() - t0), "brillo": round(brillo, 3)}), flush=True)
    torch.cuda.empty_cache()
""", encoding="utf-8")
limite = T_INICIO + LIMITE_HORAS * 3600 - 25 * 60   # deja ~25 min para voces, música y empaquetar
def repartir(TT):
  trabajos = []
  if len(MODELOS) == 1 or sum(1 for m in MODELOS if TT[m]) == 1:
    m = next((m for m in MODELOS if TT[m]), MODELOS[0])
    if m == "wan2.2-5b" and vram < 22:   # descarga por bloques: cada proceso ocupa ~15 GB de RAM; dos a la vez agotan los 30 GB de Kaggle
        trabajos = [(m, 0, TT[m])]
    else:
        trabajos = [(m, i, TT[m][i::NGPU]) for i in range(NGPU)]
  else:                   # comparación: un modelo por GPU
    trabajos = [(m, i % NGPU, TT[m]) for i, m in enumerate(MODELOS)]
  return trabajos
def correr(TT, etiqueta="planos"):
  trabajos = repartir(TT)
  procs, logs = [], []
  for m, gpu, tareas in trabajos:
    if not tareas: continue
    w, h = dims(m)
    job = {"modelo": m, "repo": SPECS[m][0], "fps": SPECS[m][1], "frames": SPECS[m][2], "shift": SPECS[m][4], "w": w, "h": h, "gpu": gpu,
           "dtype": DTYPE, "steps": STEPS, "guia": GUIA, "limite": limite, "tareas": tareas, "offload": m == "wan2.2-5b" and vram < 22}
    jf = OUT.parent / f"_job_{m}_{gpu}.json"; json.dump(job, open(jf, "w"))
    lf = open(OUT.parent / f"_log_{m}_{gpu}.txt", "a"); logs.append(lf.name)
    procs.append((m, gpu, subprocess.Popen([sys.executable, str(WORKER), str(jf)], stdout=lf, stderr=subprocess.STDOUT)))
    print(f"▶ {etiqueta}: {m} en GPU {gpu}: {len(tareas)}")
    if len(trabajos) > 1: time.sleep(60)   # escalona la carga de modelos (RAM del sistema)
  while any(p.poll() is None for _, _, p in procs):
    time.sleep(60)
    print(f"⏳ {time.strftime('%H:%M')} · {etiqueta}: {sum(1 for _, _, ts in trabajos for t in ts if os.path.exists(t['out']))}/{sum(len(ts) for _, _, ts in trabajos)}", flush=True)
  for (m, gpu, p), lf in zip(procs, logs):
    tail = open(lf).read()[-1500:]
    print(f"— {m} GPU {gpu} · código {p.returncode}\n{tail}")
    if p.returncode not in (0, None) and "OutOfMemory" not in tail and not any(os.path.exists(t["out"]) for _, _, ts in trabajos for t in ts):
        avisar("error", f"{m}: el generador falló"); raise RuntimeError(f"{m} falló en GPU {gpu}:\n{tail[-800:]}")
def elegir(destino, candidatas, identidad):
    vivas = [c for c in candidatas if os.path.exists(c)]
    if not vivas: return
    if globals().get("MT_OK"):
        cfg = {"modelos": MTM, "candidatas": vivas, "identidad": str(identidad) if identidad and os.path.exists(identidad) else None, "salida": str(destino) + ".eleccion.json"}
        json.dump(cfg, open("/tmp/elegir.json", "w")); r = sh(f"{MT_PY} /tmp/control_rostros.py --elegir /tmp/elegir.json", check=False)
        mejor = json.load(open(cfg["salida"]))[0]["img"] if r.returncode == 0 and os.path.exists(cfg["salida"]) else vivas[0]
        if r.returncode: print("⚠️ elección automática no disponible:", (r.stderr or "")[-300:])
    else:
        mejor = vivas[0]
    shutil.copy(mejor, destino); print("🖼  referencia:", pathlib.Path(destino).name, "←", pathlib.Path(mejor).name)
# Fase 1 · referencias fijas (identidad) y primeros cuadros de los primeros planos
if any(IMG_TAREAS.values()):
    correr(IMG_TAREAS, "imágenes de referencia")
for destino, cands, ident in [e for e in ELEGIR if e[2] is None] + [e for e in ELEGIR if e[2] is not None]:
    if not destino.exists(): elegir(destino, cands, ident)
for m in MODELOS:   # sin primer cuadro elegido → texto a video (nunca se detiene la producción por eso)
    for t in TAREAS[m]:
        if t.get("imagen") and not os.path.exists(t["imagen"]): t.pop("imagen")
# Fase 2 · planos de video
correr(TAREAS, "planos")
# Fase 3 · control de identidad y estabilidad facial; regenera SOLO los planos defectuosos (otra semilla), conserva los correctos
QA_PATH = OUT / "control_rostros.json"
QA = json.load(open(QA_PATH, encoding="utf-8")) if QA_PATH.exists() else {}
INTENTOS = OUT.parent / "_intentos"; INTENTOS.mkdir(exist_ok=True)
def esperados(s, k):
    return [plan(s, k)["foco"]] if es_primer_plano(s, k) else [c for c in s.get("characters", []) if c in CH]
def plano_qa(f, s, k, hoja=None):
    """Qué debe cumplir un plano: identidad OFICIAL del personaje + parecido con el primer cuadro de su escena + una sola cara en primeros planos."""
    d = {"video": str(f), "esperados": esperados(s, k), "id": f.name, "unica": es_primer_plano(s, k)}
    if es_primer_plano(s, k): d["escena"] = f"e{int(s['n']):03d}_{s['hash']}_{plan(s, k)['foco']}"
    if hoja: d["hoja"] = hoja
    return d
def medir(lista):
    if not lista: return {}
    refs = {f.stem: str(f) for f in REF.glob("*.png")}   # identidades oficiales + primeros cuadros de cada escena
    cfg = {"modelos": MTM, "refs": refs, "planos": lista, "salida": "/tmp/qa.json"}
    json.dump(cfg, open("/tmp/qa_cfg.json", "w")); r = sh(f"{MT_PY} /tmp/control_rostros.py /tmp/qa_cfg.json", check=False)
    print((r.stdout or "")[-2500:])
    if r.returncode: print("⚠️ control de rostros:", (r.stderr or "")[-600:]); return {}
    return json.load(open("/tmp/qa.json", encoding="utf-8"))
if CONTROL_ROSTROS and globals().get("MT_OK"):
    for ronda in range(int(REINTENTOS_ROSTRO) + 1):
        pend = [(DIRS[choice] / f"e{int(s['n']):03d}_p{k+1:02d}_{s['hash']}.mp4", s, k) for s in PKG["scenes"] for k in range(int(s.get("shots", 1)))]
        firma = lambda f: f"{f.stat().st_size}-{int(f.stat().st_mtime)}"
        pend = [(f, s, k) for f, s, k in pend if f.exists() and QA.get(f.name, {}).get("bytes") != firma(f)]
        res = medir([plano_qa(f, s, k) for f, s, k in pend])
        for f, s, k in pend:
            if f.name in res:
                r = res[f.name]; prev = QA.get(f.name, {})
                QA[f.name] = {**{x: r[x] for x in ("aprobado", "puntaje", "identidad_media", "identidad_min", "cuadros_con_defecto", "tramos_defectuosos", "problemas", "cara_presente")},
                              "bytes": firma(f), "intentos": prev.get("intentos", 0), "mejor": prev.get("mejor"), "rehecho": prev.get("rehecho", 0)}
        json.dump(QA, open(QA_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
        rehacer = {m: [] for m in MODELOS}
        for s in PKG["scenes"]:
            for k in range(int(s.get("shots", 1))):
                f = DIRS[choice] / f"e{int(s['n']):03d}_p{k+1:02d}_{s['hash']}.mp4"; q = QA.get(f.name)
                pedido = int(plan(s, k).get("rehacer", 0))   # rechazado en tu revisión (panel de aprobación) → se rehace solo este plano
                if q is not None and f.exists() and pedido > q.get("rehecho", 0):
                    q.update({"aprobado": False, "problemas": (q.get("problemas") or []) + ["rechazado en la revisión"], "rehecho": pedido, "intentos": min(q.get("intentos", 0), int(REINTENTOS_ROSTRO) - 1)})
                if not q or q["aprobado"] or not f.exists(): continue
                # guarda este intento; «mejor» apunta al de mayor puntaje
                cop = INTENTOS / f"{f.stem}.i{q['intentos']}.mp4"; shutil.copy(f, cop)
                QA[cop.name] = dict(q); q["mejor"] = cop.name if (not q.get("mejor") or q["puntaje"] >= QA.get(q["mejor"], {}).get("puntaje", -9)) else q["mejor"]
                if q["intentos"] >= int(REINTENTOS_ROSTRO) or time.time() > limite - 3 * 3600:
                    mejor = INTENTOS / q["mejor"]
                    if mejor.exists():
                        shutil.copy(mejor, f); q.update({x: v for x, v in QA.get(q["mejor"], {}).items() if x not in ("mejor", "intentos", "bytes")}); q["bytes"] = firma(f)
                    q["revisar"] = True; print(f"🔎 {f.name}: queda el mejor intento para tu revisión visual ({', '.join(q['problemas'])})"); continue
                t = next((t for t in TODAS.get(choice, []) if t["out"] == str(f)), None)
                if not t or not os.path.exists(t["emb"]):
                    print(f"⚠️ {f.name}: no se pudo preparar la regeneración"); continue
                t = {**t, "seed": int(t.get("seed", 1)) + 7919 * (q["intentos"] + 1)}
                if es_primer_plano(s, k) and choice in I2V:
                    im = REF / f"e{int(s['n']):03d}_{s['hash']}_{plan(s, k)['foco']}.png"
                    if im.exists(): t["imagen"] = str(im)
                q["intentos"] += 1; f.unlink(); (OUT.parent / "_sin_labios" / f.name).unlink(missing_ok=True); (OUT.parent / "_completos" / f.name).unlink(missing_ok=True)
                print(f"♻️  {f.name}: se regenera solo este plano (intento {q['intentos']}) — {', '.join(q['problemas'])}")
                rehacer[choice].append(t)
        json.dump(QA, open(QA_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
        if not any(rehacer.values()): break
        correr(rehacer, "regenerar planos defectuosos")
elif CONTROL_ROSTROS:
    print("⚠️ Control de rostros no disponible en esta sesión: el video NO se marcará como aprobado por calidad facial.")
for m, d in DIRS.items():
    if d == OUT: continue
    w, h = dims(m)
    cl = [{"file": f.name} for s in PKG["scenes"] for f in sorted(d.glob(f"e{int(s['n']):03d}_p*_{s['hash']}.mp4"))]
    json.dump({"format": "yoha-clips", "version": 2, "generator": f"{m} · {w}x{h}@{SPECS[m][1]}fps · {STEPS} pasos", "clips": cl}, open(d / "manifest.json", "w"), indent=1)
    json.dump({"completo": True, "planos_hechos": len(cl), "planos_total": len(cl), "w": w, "h": h, "fps": SPECS[m][1], "modelo": m, "comparacion": True}, open(d / "estado_sesion.json", "w"))
print("✅ Generación terminada")

## 7 · Voces (Kokoro) en español e inglés, alineadas a cada escena

In [ ]:
import soundfile as sf, numpy as np
SR = 24000
def dur(p): r = subprocess.run(["ffprobe","-v","error","-show_entries","format=duration","-of","csv=p=0",str(p)],capture_output=True,text=True); return float(r.stdout.strip() or 0)
def shots_of(s): return sorted(OUT.glob(f"e{int(s['n']):03d}_p*_{s['hash']}.mp4"))
def voice_track(lang):
    import soundfile as _sf
    key = "dialogue" if lang == "es" else "dialogueEn"
    pend = []  # (texto, voz) de todas las escenas → una sola llamada al entorno de voces
    for s in PKG["scenes"]:
        nar = s.get("narration") if lang == "es" else s.get("narrationEn")
        for cid, text in ([(None, nar)] if nar else []) + [(x.get("character"), x.get("text")) for x in s.get(key, []) if x.get("text")]:
            c = CH.get(cid, {}); pend.append({"text": text, "voice": (c.get("voice") or "ef_dora") if lang == "es" else (c.get("voiceEn") or "af_heart")})
    if not pend: return []
    tdir = OUT.parent / f"_tts_{lang}"; tdir.mkdir(exist_ok=True)
    json.dump(pend, open(tdir / "lineas.json", "w", encoding="utf-8"), ensure_ascii=False)
    (tdir / "hablar.py").write_text("""
import json, sys, numpy as np, soundfile as sf
from kokoro import KPipeline
d = sys.argv[1]; tts = KPipeline(lang_code=sys.argv[2], repo_id="hexgrad/Kokoro-82M")
for i, x in enumerate(json.load(open(d + "/lineas.json", encoding="utf-8"))):
    try: a = np.concatenate([np.asarray(t, np.float32) for _, _, t in tts(x["text"], voice=x["voice"])])
    except Exception as e:
        print("voz alternativa para", x["voice"], e); a = np.concatenate([np.asarray(t, np.float32) for _, _, t in tts(x["text"], voice="ef_dora" if sys.argv[2] == "e" else "af_heart")])
    sf.write(f"{d}/l{i:04d}.wav", a, 24000)
print("ok", i + 1)
""", encoding="utf-8")
    motor = "kokoro"
    if VOZ_MOTOR == "chatterbox" and globals().get("CBX_OK"):
        # 1) muestra de voz por personaje con Kokoro (da a cada uno un timbre distinto); 2) Chatterbox actúa cada línea con esa voz
        EXAG = {"neutral": .45, "feliz": .55, "risa": .65, "triste": .55, "llanto": .7, "enfado": .7, "sorpresa": .6, "miedo": .6, "amor": .5, "celos": .6, "determinacion": .55}
        voces = sorted({x["voice"] for x in pend})
        rdir = tdir / "ref"; rdir.mkdir(exist_ok=True)
        json.dump([{"text": "Hola. Esta es mi voz. Hoy les voy a contar una historia de esfuerzo, de familia y de sueños que se cumplen, paso a paso.", "voice": v} for v in voces], open(rdir / "lineas.json", "w", encoding="utf-8"), ensure_ascii=False)
        (rdir / "hablar.py").write_text((tdir / "hablar.py").read_text(encoding="utf-8"), encoding="utf-8")
        r1 = sh(f"{TTS_PY} {rdir / 'hablar.py'} {rdir} {'e' if lang == 'es' else 'a'}", check=False)
        emos = []
        for s in PKG["scenes"]:
            nar = s.get("narration") if lang == "es" else s.get("narrationEn")
            emos += (["determinacion"] if nar else []) + [x.get("emotion", "neutral") for x in s.get(key, []) if x.get("text")]
        for i, x in enumerate(pend):
            x["ref"] = str(rdir / f"l{voces.index(x['voice']):04d}.wav") if r1.returncode == 0 else None
            x["exag"], x["cfg"] = EXAG.get(emos[i] if i < len(emos) else "neutral", .5), .3   # cfg bajo: ritmo pausado y natural
        json.dump(pend, open(tdir / "lineas.json", "w", encoding="utf-8"), ensure_ascii=False)
        (tdir / "actuar.py").write_text("""
import json, sys, torch, soundfile as sf
from chatterbox.mtl_tts import ChatterboxMultilingualTTS
d, lang = sys.argv[1], sys.argv[2]
m = ChatterboxMultilingualTTS.from_pretrained(device="cuda" if torch.cuda.is_available() else "cpu")
for i, x in enumerate(json.load(open(d + "/lineas.json", encoding="utf-8"))):
    kw = {"language_id": lang, "exaggeration": x["exag"], "cfg_weight": x["cfg"]}
    if x.get("ref"): kw["audio_prompt_path"] = x["ref"]
    for j in range(3):   # tres tomas: se queda la que mejor se entiende y con ritmo humano
        torch.manual_seed(1000 + 17 * i + j)
        w = m.generate(x["text"], **kw)
        sf.write(f"{d}/l{i:04d}_t{j}.wav", w.squeeze().cpu().numpy(), m.sr)
print("ok")
""", encoding="utf-8")
        r = sh(f"{CBX_PY} {tdir / 'actuar.py'} {tdir} {lang}", check=False)
        if r.returncode == 0: motor = "chatterbox"
        else: print("⚠️ Chatterbox falló, se usa Kokoro:", (r.stderr or r.stdout)[-500:])
    if motor == "kokoro":
        r = sh(f"{TTS_PY} {tdir / 'hablar.py'} {tdir} {'e' if lang == 'es' else 'a'}", check=False)
        if r.returncode: raise RuntimeError("Voces: " + (r.stderr or r.stdout)[-600:])
    def _leer(f):
        a, sr = _sf.read(f, dtype="float32")
        if a.ndim > 1: a = a.mean(axis=1)
        if sr != SR: a = np.interp(np.linspace(0, len(a), int(len(a) * SR / sr), endpoint=False), np.arange(len(a)), a).astype(np.float32)
        return a
    def _recortar(a):   # quita silencios de los extremos (deja 60 ms de aire)
        e = np.convolve(np.abs(a), np.ones(240) / 240, "same"); v = np.where(e > max(1e-4, e.max() * 0.04))[0]
        return a[max(0, v[0] - 1440): v[-1] + 1440] if len(v) else a
    def _nivelar(a, rms_db=-20.0):   # mismo volumen percibido para todos los personajes, sin saturar
        voz = a[np.abs(a) > np.abs(a).max() * 0.05] if np.abs(a).max() > 0 else a
        g = 10 ** (rms_db / 20) / max(1e-6, float(np.sqrt(np.mean(voz ** 2))))
        return np.clip(a * min(g, 8.0), -0.97, 0.97).astype(np.float32)
    def _silabas(t):
        import re as _re, unicodedata as _u
        t = _u.normalize("NFKD", t.lower()); return max(1, len(_re.findall(r"[aeiouáéíóúü]+", t)))
    _wm = None
    def _oir(a):
        nonlocal _wm
        try:
            if _wm is None:
                from faster_whisper import WhisperModel
                _wm = WhisperModel("small", device="cuda" if torch.cuda.is_available() else "cpu", compute_type="float16" if torch.cuda.is_available() else "int8")
            a16 = np.interp(np.linspace(0, len(a), int(len(a) * 16000 / SR), endpoint=False), np.arange(len(a)), a).astype(np.float32)
            return " ".join(x.text for x in _wm.transcribe(a16, language=lang)[0])
        except Exception as e:
            print("⚠️ Whisper (tomas):", e); return None
    def _palabras(x):
        import re as _re, unicodedata as _u
        x = _u.normalize("NFKD", x.lower()); x = "".join(c for c in x if not _u.combining(c)); return set(w for w in _re.findall(r"[a-zñ]+", x) if len(w) >= 3)
    elegidas, informe = [], []
    for i, x in enumerate(pend):
        tomas = sorted(tdir.glob(f"l{i:04d}_t*.wav")) or [tdir / f"l{i:04d}.wav"]
        mejor = None
        for f in tomas:
            a = _recortar(_leer(f)); dur_s = len(a) / SR; ritmo = _silabas(x["text"]) / max(0.2, dur_s)
            oido = _oir(a) if len(tomas) > 1 else None
            esperadas = _palabras(x["text"]); coincide = (len(esperadas & _palabras(oido)) / max(1, len(esperadas))) if oido is not None else 1.0
            puntaje = coincide - 0.15 * max(0.0, ritmo - 6.0) - 0.15 * max(0.0, 3.0 - ritmo)   # 3–6 sílabas/s ≈ habla natural en español
            r = {"linea": i, "texto": x["text"], "toma": f.name, "segundos": round(dur_s, 2), "silabas_s": round(ritmo, 2), "whisper": oido, "coincidencia": round(coincide, 2), "puntaje": round(puntaje, 3)}
            if mejor is None or puntaje > mejor[0]: mejor = (puntaje, a, r)
        elegidas.append(_nivelar(mejor[1])); informe.append(mejor[2])
        print(f"   🎙 línea {i+1}: {mejor[2]['toma']} · {mejor[2]['segundos']} s · {mejor[2]['silabas_s']} síl/s · entiende {mejor[2]['coincidencia']:.0%}")
    json.dump(informe, open(OUT / f"voces_informe_{lang}.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    audios = iter(elegidas)
    print(f"🎙  Voces ({lang}) con {motor}")
    track, t, subs = np.zeros(0, np.float32), 0.0, []
    largos = list(elegidas)
    pos_linea = 0
    for s in PKG["scenes"]:   # 1) cuánto debe durar cada plano según lo que se dice en él
        nar = (s.get("narration") if lang == "es" else s.get("narrationEn"))
        idx = ([None] if nar else []) + [j for j, x in enumerate(s.get(key, [])) if x.get("text")]
        lens = {j: len(largos[pos_linea + n]) / SR for n, j in enumerate(idx)}; pos_linea += len(idx)
        for k, p in enumerate(shots_of(s)):
            pl = plan(s, k); full = dur(p)
            if pl.get("lineas"):
                hablado = sum(lens.get(j, 0) for j in pl["lineas"]) + 0.45 * max(0, len(pl["lineas"]) - 1)
                CORTES[p.name] = round(min(full, max(2.4, 0.4 + hablado + 0.8)), 3)
            elif es_primer_plano(s, k):   # plano de reacción: lo justo para leer la emoción
                CORTES[p.name] = round(min(full, 3.2), 3)
    for s in PKG["scenes"]:
        d = sum(CORTES.get(p.name, dur(p)) for p in shots_of(s)) or s.get("seconds", 5)
        lines = ([(None, s.get("narration") if lang == "es" else s.get("narrationEn", ""), None)] if (s.get("narration") if lang == "es" else s.get("narrationEn")) else []) + \
                [(x.get("character"), x.get("text"), j) for j, x in enumerate(s.get(key, [])) if x.get("text")]
        inicios, acc = [], 0.0
        for p in shots_of(s): inicios.append(t + acc); acc += CORTES.get(p.name, dur(p))
        pos = t + 0.4
        for cid, text, j in lines:
            c = CH.get(cid, {}); audio = next(audios)
            kpl = next((k for k in range(len(inicios)) if j is not None and j in plan(s, k).get("lineas", [])), None)
            if kpl is not None: pos = max(pos, inicios[kpl] + 0.4)   # la frase arranca dentro del primer plano de quien la dice
            a0 = int(pos * SR); a1 = a0 + len(audio)
            if len(track) < a1: track = np.pad(track, (0, a1 - len(track)))
            track[a0:a1] += audio
            subs.append((pos, pos + len(audio) / SR, (c.get("name", "") + ": " if c else "") + text)); pos += len(audio) / SR + 0.45   # pausa humana entre frases
        t += d
    if track.size: sf.write(OUT / f"voces_{lang}.wav", np.clip(track, -1, 1), SR)
    srt = "\n".join(f"{i+1}\n{fmt(a)} --> {fmt(b)}\n{x}\n" for i, (a, b, x) in enumerate(subs))
    (OUT / f"subtitulos_{lang}.srt").write_text(srt, encoding="utf-8")
    print(f"✅ voces_{lang}.wav · {track.size/SR:.1f}s de voz · video {t:.1f}s"); return subs
def fmt(x): ms = int(x * 1000); return f"{ms//3600000:02d}:{ms//60000%60:02d}:{ms//1000%60:02d},{ms%1000:03d}"
CORTES = {}
def aplicar_cortes():
    """Cada plano dura lo que pide su diálogo; el original completo se guarda en _completos/ (nunca se pierde material)."""
    COMP = OUT.parent / "_completos"; COMP.mkdir(exist_ok=True)
    for name, d in CORTES.items():
        p = OUT / name; orig = COMP / name
        if not p.exists(): continue
        if not orig.exists(): shutil.copy(p, orig)
        if abs(dur(p) - d) > 0.05:
            tmp = OUT.parent / f"_corte_{name}"
            sh(f"ffmpeg -v error -y -i {orig} -t {d:.3f} -c:v libx264 -crf 14 -preset slow -pix_fmt yuv420p -an {tmp}"); os.replace(tmp, p)
            (OUT.parent / "_sin_labios" / name).unlink(missing_ok=True)
    json.dump(CORTES, open(OUT / "cortes.json", "w"), indent=1)
SUBS = {}
if VOCES:
    for lang in PKG.get("languages", ["es"]):
        if lang == "en" and not any(s.get("dialogueEn") or s.get("narrationEn") for s in PKG["scenes"]):
            print("Inglés: el guion no trae diálogos en inglés (pídelos al guionista IA)."); continue
        try:
            SUBS[lang] = voice_track(lang)
            if lang == PKG.get("languages", ["es"])[0]: aplicar_cortes()
        except Exception as e:  # sin voces la Productora NO monta un video mudo (lo bloquea y avisa)
            print(f"⚠️ No se generaron las voces ({lang}): {e}")

## 7b · Música original (ACE-Step, Apache-2.0) según el ánimo de la producción

In [ ]:
if MUSICA and globals().get("ACE_OK"):
    total_s = sum(sum(dur(p) for p in shots_of(s)) or s.get("seconds", 5) for s in PKG["scenes"])   # los planos ya están recortados al diálogo
    MOODS = {"inspiradora": "cinematic orchestral film score, emotional piano, warm strings, uplifting, inspirational, hopeful, instrumental",
             "dramatica": "dramatic cinematic orchestral score, tense strings, deep drums, emotional, instrumental",
             "romantica": "romantic cinematic piano and strings, tender, instrumental", "suspenso": "dark suspense film score, low strings, pulses, instrumental",
             "alegre": "bright latin acoustic guitar and light percussion, happy, instrumental", "neutra": "soft cinematic ambient piano and strings, instrumental"}
    mood = MOODS.get(PKG.get("musicMood", "neutra"), MOODS["neutra"])
    mdir = OUT.parent / "_musica"; mdir.mkdir(exist_ok=True)
    (mdir / "componer.py").write_text("""
import sys, inspect, torchaudio, soundfile as _sf
def _save(path, wav, sample_rate=48000, *a, **k):   # torchaudio >= 2.9 exige torchcodec; soundfile guarda igual (sin pérdida)
    _sf.write(str(path), wav.detach().float().cpu().numpy().T, int(sample_rate))
torchaudio.save = _save
from acestep.pipeline_ace_step import ACEStepPipeline
dur, prompt, out = float(sys.argv[1]), sys.argv[2], sys.argv[3]
p = ACEStepPipeline(dtype="float32", torch_compile=False, cpu_offload=True, overlapped_decode=True)
kw = dict(audio_duration=dur, prompt=prompt, lyrics="[instrumental]", infer_step=60, guidance_scale=15.0, scheduler_type="euler", cfg_type="apg",
          omega_scale=10.0, manual_seeds=[2026], guidance_interval=0.5, guidance_interval_decay=0.0, min_guidance_scale=3.0,
          use_erg_tag=True, use_erg_lyric=False, use_erg_diffusion=True, oss_steps=[], guidance_scale_text=0.0, guidance_scale_lyric=0.0, save_path=out)
ok = set(inspect.signature(p.__call__).parameters)
p(**{k: v for k, v in kw.items() if k in ok})
print("ok")
""", encoding="utf-8")
    r = sh(f"cd {mdir} && {ACE_PY} componer.py {min(240, max(30, total_s))} '{mood}' {OUT / 'musica.wav'}", check=False)
    print("✅ musica.wav (original, Apache-2.0)" if (OUT / "musica.wav").exists() else "⚠️ Música no generada: " + (r.stderr or r.stdout)[-600:])

## 8 · Subtítulos con Whisper · labios sincronizados (MuseTalk 1.5) · control final de rostros

In [ ]:
def _whisper():
    from faster_whisper import WhisperModel
    wm = WhisperModel("small", device="cuda" if torch.cuda.is_available() else "cpu", compute_type="float16" if torch.cuda.is_available() else "int8")
    a, sr = sf.read(OUT / "voces_es.wav", dtype="float32")
    a = np.interp(np.linspace(0, len(a), int(len(a) * 16000 / sr), endpoint=False), np.arange(len(a)), a).astype(np.float32)
    segs, _ = wm.transcribe(a, language="es")
    (OUT / "subtitulos_whisper_es.srt").write_text("\n".join(f"{i+1}\n{fmt(s.start)} --> {fmt(s.end)}\n{s.text.strip()}\n" for i, s in enumerate(segs)), encoding="utf-8")
    print("✅ subtitulos_whisper_es.srt (verificación de lo que realmente se oye)")
if SUBTITULOS_WHISPER and (OUT / "voces_es.wav").exists():
    try: _whisper()
    except Exception as e: print("⚠️ Whisper no disponible:", e)

LABIOS_PLANOS = []   # primeros planos con diálogo
if LABIOS_SINCRONIZADOS and (OUT / "voces_es.wav").exists():
    reg_path = OUT / "labios.json"; reg = json.load(open(reg_path)) if reg_path.exists() else {}
    SIN = OUT.parent / "_sin_labios"; SIN.mkdir(exist_ok=True)
    jobs, t = [], 0.0
    for s in PKG["scenes"]:
        for k, p in enumerate(shots_of(s)):
            d = dur(p); pl = plan(s, k)
            if es_primer_plano(s, k):
                LABIOS_PLANOS.append(p.name)
                seg = OUT.parent / f"_a_{p.stem}.wav"
                if pl.get("lineas"):   # su propia voz en su propio plano
                    sh(f"ffmpeg -v error -y -ss {t:.3f} -t {d:.3f} -i {OUT / 'voces_es.wav'} -ar 16000 -ac 1 {seg}")
                else:                  # plano de reacción: escucha → silencio (la boca queda en reposo aunque la otra persona hable)
                    sh(f"ffmpeg -v error -y -f lavfi -i anullsrc=r=16000:cl=mono -t {d:.3f} {seg}")
                orig = SIN / p.name
                if not orig.exists(): shutil.copy(p, orig)
                firma = hashlib.sha1(seg.read_bytes() + str(orig.stat().st_size).encode()).hexdigest()[:12]
                if reg.get(p.name) != firma:
                    src = orig
                    if abs(FPS - 24) > 0.5:   # 16 → 24 fps antes de mover los labios (más fluido)
                        src = OUT.parent / f"_24_{p.name}"
                        sh(f"ffmpeg -v error -y -i {orig} -vf minterpolate=fps=24:mi_mode=mci:mc_mode=aobmc:vsbmc=1 -c:v libx264 -crf 14 -pix_fmt yuv420p {src}")
                    jobs.append({"video": str(src), "audio": str(seg), "salida": str(p), "firma": firma, "escucha": not pl.get("lineas")})
            t += d
    if jobs and globals().get("MT_OK"):
        json.dump({"musetalk": MT_REPO, "modelos": MTM, "trabajos": jobs, "informe": "/tmp/labios_nuevo.json"}, open("/tmp/labios_jobs.json", "w"))
        r = sh(f"cd {MT_REPO} && {MT_PY} /tmp/labios_musetalk.py /tmp/labios_jobs.json", check=False)
        print((r.stdout or "")[-3000:])
        if r.returncode: print("⚠️ Labios:", (r.stderr or "")[-1500:])
        nuevo = json.load(open("/tmp/labios_nuevo.json")) if os.path.exists("/tmp/labios_nuevo.json") else []
        hechos = {x["plano"] for x in nuevo}
        for j in jobs:
            if pathlib.Path(j["salida"]).name in hechos: reg[pathlib.Path(j["salida"]).name] = j["firma"]
        json.dump(reg, open(reg_path, "w"))
        previo = json.load(open(OUT / "labios_informe.json")) if (OUT / "labios_informe.json").exists() else []
        inf = {x["plano"]: x for x in previo}; inf.update({x["plano"]: x for x in nuevo})
        json.dump(list(inf.values()), open(OUT / "labios_informe.json", "w"), ensure_ascii=False, indent=1)
        print(f"👄 Labios sincronizados en {len(hechos)}/{len(jobs)} primeros planos")
    elif jobs:
        print("⚠️ Labios no disponibles en esta sesión: estos planos quedan SIN sincronizar y la Productora no los aprobará:", [pathlib.Path(j['salida']).name for j in jobs])
# Control final (después de los labios): informe por plano + hojas de contacto para la revisión visual
if CONTROL_ROSTROS and globals().get("MT_OK"):
    REV = OUT / "revision"; REV.mkdir(exist_ok=True)
    REF = OUT / "referencias"; refs = {f.stem: str(f) for f in REF.glob("*.png")}
    pl_ = []
    for s in PKG["scenes"]:
        for k, p in enumerate(shots_of(s)):
            pl_.append(plano_qa(p, s, k, str(REV / f"{p.stem}.jpg")))
    json.dump({"modelos": MTM, "refs": refs, "planos": pl_, "salida": str(OUT / "control_final.json")}, open("/tmp/qa_final.json", "w"))
    r = sh(f"{MT_PY} /tmp/control_rostros.py /tmp/qa_final.json", check=False)
    print((r.stdout or "")[-3000:] if r.returncode == 0 else "⚠️ control final: " + (r.stderr or "")[-600:])

## 9 · Escalado opcional (escalado, no 4K nativo) · empaquetar · avisar al iPhone

In [ ]:
import zipfile
from IPython.display import Video, display
clips = [p for s in PKG["scenes"] for p in shots_of(s)]
if ESCALAR_A != "ninguno":
    short = 1080 if ESCALAR_A == "1080p" else 2160
    for p in clips:
        tmp = p.with_name(p.stem + ".up.mp4")
        sh(f"ffmpeg -v error -y -i {p} -vf scale={'-2:'+str(short) if W >= H else str(short)+':-2'}:flags=lanczos,unsharp=5:5:0.4 -c:v libx264 -crf 18 -pix_fmt yuv420p {tmp}")
        tmp.replace(p)
for p in clips[:3]: display(Video(str(p), embed=True, width=420))
def sub_for(s, k):
    lines = [d.get("text", "") for d in s.get("dialogue", [])]
    if not lines: return s.get("narration", "") if k == 0 else ""
    per = max(1, -(-len(lines) // int(s.get("shots", 1))))
    return " ".join(lines[k * per:(k + 1) * per])
manifest = {"format": "yoha-clips", "version": 2, "generator": f"{choice} · {W}x{H}@{FPS}fps · {STEPS} pasos", "upscaled": ESCALAR_A, "production": PKG.get("id"), "title": PKG.get("title"),
            "clips": [], "narration": "voces_es.wav" if (OUT / "voces_es.wav").exists() else None, "narrationEn": "voces_en.wav" if (OUT / "voces_en.wav").exists() else None}
for s in PKG["scenes"]:
    for k, p in enumerate(shots_of(s)):
        manifest["clips"].append({"file": p.name, "scene": s["id"], "title": PKG.get("title", "") if (s["n"] == 1 and k == 0) else "", "subtitle": sub_for(s, k), "seconds": round(dur(p), 2)})
json.dump(manifest, open(OUT / "manifest.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
total = len([p for s in PKG["scenes"] for p in range(int(s.get('shots', 1)))])
json.dump({"completo": len(clips) >= total, "planos_hechos": len(clips), "planos_total": total, "titulo": PKG.get("title"), "w": W, "h": H, "fps": FPS, "modelo": choice}, open(OUT / "estado_sesion.json", "w", encoding="utf-8"), ensure_ascii=False)
if YOHA_AUTO:
    print(f"✅ Sesión terminada · {len(clips)}/{total} planos · la Productora de Yoha Video monta el video final.")
else:
    zpath = OUT.parent / "yoha_clips.zip"
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_STORED) as z:
        for f in [*clips, *OUT.glob("*.wav"), *OUT.glob("*.srt"), OUT / "manifest.json", OUT / "report.json", OUT / "guion_ia.json"]:
            if f.exists() and not f.name.startswith("_a_"): z.write(f, f"yoha_clips/{f.name}")
    print(f"✅ {zpath} · {zpath.stat().st_size/1e6:.1f} MB · {len(clips)}/{total} planos")
    if len(clips) >= total: avisar("ready", f"{PKG.get('title')}: {len(clips)} planos listos.")
    else: avisar("stopped", f"{len(clips)}/{total} planos. Ejecuta de nuevo la celda 6 en otra sesión.")
    try:
        from google.colab import files as gf  # type: ignore
        gf.download(str(zpath))
    except ImportError:
        print("Kaggle: panel Output → yoha_clips.zip → Download. Ábrelo en Yoha Video → Editor.")